# GGA Network Training Example

This notebook demonstrates how to train a GGA-level exchange-correlation functional using xcquinox.

## Outline

1. **Setup**: Import libraries and define helper functions
2. **Pre-training**: Train exchange (X) and correlation (C) networks on reference functional data (PBE)
3. **Training Part I**: Train on total energies of atoms + atomization energies of small molecules
4. **Training Part II**: Include density error in the loss function

All calculations use PySCF-AD (autodifferentiable PySCF) as the calculation driver.

---
## 1. Setup

In [1]:
# Core imports
import numpy as np
import jax
import jax.numpy as jnp
import equinox as eqx
import optax
import matplotlib.pyplot as plt

# PySCF imports
import pyscf
from pyscf import gto, dft, scf

# PySCF-AD imports (autodifferentiable)
import pyscfad
from pyscfad import gto as gto_ad
from pyscfad import dft as dft_ad

# xcquinox imports
import xcquinox as xce
from xcquinox.train import Pretrainer, Optimizer
from xcquinox.utils import lda_x, pw92c_unpolarized
from functools import partial

# ASE for molecular structures
from ase import Atoms

# Enable 64-bit precision for stability
jax.config.update("jax_enable_x64", True)

print(f"JAX version: {jax.__version__}")
print(f"PySCF version: {pyscf.__version__}")
print(f"PySCF-AD version: {pyscfad.__version__}")
print(f"Using device: {jax.devices()[0]}")

JAX version: 0.7.0
PySCF version: 2.11.0
PySCF-AD version: 0.1.11
Using device: TFRT_CPU_0


In [2]:
# Configuration - keep computations affordable
BASIS = 'sto-3g'           # Small basis for speed
GRID_LEVEL = 1             # Coarse grid for speed
REFERENCE_XC = 'PBE'       # Reference functional for pre-training

# Use CPU to avoid GPU memory issues
CPU_DEVICE = jax.devices('cpu')[0]

# Spin configurations for isolated atoms
SPINS_DICT = {
    'H': 1, 'He': 0, 'Li': 1, 'Be': 0, 'B': 1, 'C': 2, 'N': 3, 'O': 2, 'F': 1, 'Ne': 0,
    'Na': 1, 'Mg': 0, 'Al': 1, 'Si': 2, 'P': 3, 'S': 2, 'Cl': 1, 'Ar': 0
}

# Checkpoint directory structure
import os
CHECKPOINT_BASE = 'checkpoints'
CHECKPOINT_DIRS = {
    # Architecture A (shallow)
    'pretrain_xnet_A': os.path.join(CHECKPOINT_BASE, '01_pretrain_xnet_A'),
    'pretrain_cnet_A': os.path.join(CHECKPOINT_BASE, '02_pretrain_cnet_A'),
    # Architecture B (deeper, standard)
    'pretrain_xnet_B': os.path.join(CHECKPOINT_BASE, '03_pretrain_xnet_B'),
    'pretrain_cnet_B': os.path.join(CHECKPOINT_BASE, '04_pretrain_cnet_B'),
    # Architecture C (deeper, self-attention)
    'pretrain_xnet_C': os.path.join(CHECKPOINT_BASE, '03c_pretrain_xnet_C_attn'),
    'pretrain_cnet_C': os.path.join(CHECKPOINT_BASE, '04c_pretrain_cnet_C_attn'),
    # Architecture D (transform, no self-attention)
    'pretrain_xnet_D': os.path.join(CHECKPOINT_BASE, '03d_pretrain_xnet_D_transform'),
    'pretrain_cnet_D': os.path.join(CHECKPOINT_BASE, '04d_pretrain_cnet_D_transform'),
    # Architecture E (transform + self-attention)
    'pretrain_xnet_E': os.path.join(CHECKPOINT_BASE, '03e_pretrain_xnet_E_transform_attn'),
    'pretrain_cnet_E': os.path.join(CHECKPOINT_BASE, '04e_pretrain_cnet_E_transform_attn'),
    # Training checkpoints (standard)
    'train_energy': os.path.join(CHECKPOINT_BASE, '05_train_energy'),
    'train_energy_density': os.path.join(CHECKPOINT_BASE, '06a_train_energy_density'),
    'train_energy_density_hw': os.path.join(CHECKPOINT_BASE, '06b_train_energy_density_hw'),
    'train_grid_density': os.path.join(CHECKPOINT_BASE, '07_train_grid_density'),
    # Training checkpoints (self-attention)
    'train_energy_attn': os.path.join(CHECKPOINT_BASE, '05a_train_energy_attn'),
    'train_energy_density_attn': os.path.join(CHECKPOINT_BASE, '06c_train_energy_density_attn'),
    'train_grid_density_attn': os.path.join(CHECKPOINT_BASE, '07a_train_grid_density_attn'),
    # Final models
    'final_models': os.path.join(CHECKPOINT_BASE, '08_final_models'),
}

# Create all checkpoint directories
for name, path in CHECKPOINT_DIRS.items():
    os.makedirs(path, exist_ok=True)
    print(f"Checkpoint dir: {path}")

print(f"\nAll checkpoints will be saved under: {CHECKPOINT_BASE}/")

Checkpoint dir: checkpoints/01_pretrain_xnet_A
Checkpoint dir: checkpoints/02_pretrain_cnet_A
Checkpoint dir: checkpoints/03_pretrain_xnet_B
Checkpoint dir: checkpoints/04_pretrain_cnet_B
Checkpoint dir: checkpoints/03c_pretrain_xnet_C_attn
Checkpoint dir: checkpoints/04c_pretrain_cnet_C_attn
Checkpoint dir: checkpoints/05_train_energy
Checkpoint dir: checkpoints/06a_train_energy_density
Checkpoint dir: checkpoints/06b_train_energy_density_hw
Checkpoint dir: checkpoints/07_train_grid_density
Checkpoint dir: checkpoints/05a_train_energy_attn
Checkpoint dir: checkpoints/06c_train_energy_density_attn
Checkpoint dir: checkpoints/07a_train_grid_density_attn
Checkpoint dir: checkpoints/08_final_models

All checkpoints will be saved under: checkpoints/


In [3]:
def create_mol(atoms_str, basis=BASIS, charge=0, spin=None):
    '''
    Create a PySCF Mole object from atom string.
    
    :param atoms_str: Atom specification (e.g., 'H 0 0 0; H 0 0 0.74')
    :param basis: Basis set name
    :param charge: Molecular charge
    :param spin: 2S (number of unpaired electrons), auto-detected for single atoms
    :return: Built PySCF Mole object
    '''
    mol = gto.Mole()
    mol.atom = atoms_str
    mol.basis = basis
    mol.charge = charge
    if spin is not None:
        mol.spin = spin
    mol.build()
    return mol


def create_mol_ad(atoms_str, basis=BASIS, charge=0, spin=None):
    '''
    Create a PySCF-AD Mole object (autodifferentiable).
    
    :param atoms_str: Atom specification (e.g., 'H 0 0 0; H 0 0 0.74')
    :param basis: Basis set name
    :param charge: Molecular charge
    :param spin: 2S (number of unpaired electrons)
    :return: Built PySCF-AD Mole object
    '''
    mol = gto_ad.Mole()  # Fixed: was gtoad, now gto_ad
    mol.atom = atoms_str
    mol.basis = basis
    mol.charge = charge
    if spin is not None:
        mol.spin = spin
    mol.build()
    return mol


def run_pbe_calculation(mol):
    '''
    Run a standard PBE DFT calculation.
    
    :param mol: PySCF Mole object
    :return: Converged mf object
    '''
    if mol.spin == 0:
        mf = dft.RKS(mol)
    else:
        mf = dft.UKS(mol)
    mf.xc = REFERENCE_XC
    mf.grids.level = GRID_LEVEL
    mf.kernel()
    return mf


def get_enhancement_factor_data(mol, mf, xorc='x'):
    '''
    Extract enhancement factor data from a converged calculation for pre-training.

    :param mol: PySCF Mole object
    :param mf: Converged mf object
    :param xorc: 'x' for exchange or 'c' for correlation
    :return: (descriptors, reference_enhancement_factors)
    '''
    ao = mf._numint.eval_ao(mol, mf.grids.coords, deriv=1)
    dm = mf.make_rdm1()

    # Handle spin - for UKS, dm has shape (2, nao, nao)
    if len(dm.shape) == 2:
        # RKS: split into alpha/beta
        dm_alpha = dm * 0.5
        dm_beta = dm * 0.5
    else:
        # UKS: already split
        dm_alpha = dm[0]
        dm_beta = dm[1]

    # Evaluate density on grid (GGA format: rho, grad_x, grad_y, grad_z)
    rho_alpha = mf._numint.eval_rho(mol, ao, dm_alpha, xctype='GGA', hermi=True)
    rho_beta = mf._numint.eval_rho(mol, ao, dm_beta, xctype='GGA', hermi=True)

    # Extract density and gradient components
    rho0_a = rho_alpha[0]  # density
    rho0_b = rho_beta[0]
    drho_a = rho_alpha[1:4]  # gradients
    drho_b = rho_beta[1:4]

    # Total density and sigma for descriptors
    rho0 = rho0_a + rho0_b
    sigma = np.sum((drho_a + drho_b)**2, axis=0)

    if xorc == 'x':
        # Exchange: Fx = exc / lda_x (spin=0 for unpolarized reference)
        xc_func = f'{REFERENCE_XC},'
        # For GGA eval_xc with spin=0, pass total density in GGA format
        rho_total = rho_alpha + rho_beta  # Total in GGA format (4, ngrids)
        exc = mf._numint.eval_xc(xc_func, rho_total, spin=0)[0]
        lda_exc = mf._numint.eval_xc('LDA_X,', rho0, spin=0)[0]
        Fxc = exc / (lda_exc + 1e-10) - 1  # Enhancement factor minus 1
    else:
        # Correlation: Fc = ec / lda_c
        xc_func = f',{REFERENCE_XC}'
        rho_total = rho_alpha + rho_beta
        exc = mf._numint.eval_xc(xc_func, rho_total, spin=0)[0]
        lda_exc = mf._numint.eval_xc(',LDA_C_PW', rho0, spin=0)[0]
        Fxc = exc / (lda_exc + 1e-10) - 1

    # Filter out low-density regions
    valid = rho0 > 1e-6
    rho0 = rho0[valid]
    sigma = sigma[valid]
    Fxc = Fxc[valid]

    # Stack as input features [rho, sigma]
    descriptors = np.stack([rho0, sigma], axis=1)

    return jnp.array(descriptors), jnp.array(Fxc)

---
## 2. Pre-training

Pre-training fits the neural network enhancement factors (Fx and Fc) to reproduce a reference functional (PBE) on a set of atoms and small molecules.

We'll explore two different network architectures:
- **Architecture A**: Shallow network (depth=2, nodes=8)
- **Architecture B**: Deeper network (depth=3, nodes=16)

In [4]:
# Define pre-training molecules (small atoms and diatomics)
pretrain_systems = [
    ('H', 1),      # Hydrogen atom
    ('C', 2),      # Carbon atom
    ('N', 3),      # Nitrogen atom
    ('O', 2),      # Oxygen atom
    ('H 0 0 0; H 0 0 0.74', 0),   # H2
    ('N 0 0 0; N 0 0 1.1', 0),     # N2
    ('O 0 0 0; O 0 0 1.21', 2),    # O2 (triplet)
]

print("Pre-training molecules:")
for atoms, spin in pretrain_systems:
    print(f"  {atoms} (spin={spin})")

Pre-training molecules:
  H (spin=1)
  C (spin=2)
  N (spin=3)
  O (spin=2)
  H 0 0 0; H 0 0 0.74 (spin=0)
  N 0 0 0; N 0 0 1.1 (spin=0)
  O 0 0 0; O 0 0 1.21 (spin=2)


In [5]:
# Collect enhancement factor data from reference calculations
print("Running reference calculations and collecting enhancement factor data...")

all_descriptors_x = []
all_Fx = []
all_descriptors_c = []
all_Fc = []

for atoms_str, spin in pretrain_systems:
    print(f"  Processing: {atoms_str}")
    mol = create_mol(atoms_str, spin=spin)
    mf = run_pbe_calculation(mol)
    
    # Exchange data
    desc_x, Fx = get_enhancement_factor_data(mol, mf, xorc='x')
    all_descriptors_x.append(desc_x)
    all_Fx.append(Fx)
    
    # Correlation data
    desc_c, Fc = get_enhancement_factor_data(mol, mf, xorc='c')
    all_descriptors_c.append(desc_c)
    all_Fc.append(Fc)

# Concatenate all data
train_desc_x = jnp.concatenate(all_descriptors_x, axis=0)
train_Fx = jnp.concatenate(all_Fx, axis=0)
train_desc_c = jnp.concatenate(all_descriptors_c, axis=0)
train_Fc = jnp.concatenate(all_Fc, axis=0)

print(f"\nCollected data shapes:")
print(f"  Exchange descriptors: {train_desc_x.shape}")
print(f"  Exchange Fx targets: {train_Fx.shape}")
print(f"  Correlation descriptors: {train_desc_c.shape}")
print(f"  Correlation Fc targets: {train_Fc.shape}")

Running reference calculations and collecting enhancement factor data...
  Processing: H
converged SCF energy = -0.464375662953948  <S^2> = 0.75  2S+1 = 2
  Processing: C
converged SCF energy = -37.2868345055596  <S^2> = 2  2S+1 = 3
  Processing: N
converged SCF energy = -53.814703131493  <S^2> = 3.75  2S+1 = 4
  Processing: O
converged SCF energy = -73.9487655565451  <S^2> = 2  2S+1 = 3
  Processing: H 0 0 0; H 0 0 0.74
converged SCF energy = -1.15207386040247
  Processing: N 0 0 0; N 0 0 1.1
converged SCF energy = -107.928182006105
  Processing: O 0 0 0; O 0 0 1.21
converged SCF energy = -148.138372397661  <S^2> = 2.0003916  2S+1 = 3.000261

Collected data shapes:
  Exchange descriptors: (33650, 2)
  Exchange Fx targets: (33650,)
  Correlation descriptors: (33650, 2)
  Correlation Fc targets: (33650,)


In [6]:
class PretrainLoss(eqx.Module):
    '''
    MSE loss for pre-training enhancement factor networks.
    '''
    def __call__(self, model, descriptors, ref_F):
        '''
        Compute MSE loss between predicted and reference enhancement factors.
        
        :param model: GGA network (GGA_FxNet_sigma or GGA_FcNet_sigma)
        :param descriptors: Input descriptors shape (N, 2) with [rho, sigma] per row
        :param ref_F: Reference enhancement factors shape (N,)
        :return: MSE loss
        '''
        # Networks expect one grid point at a time: inputs[0]=rho, inputs[1]=sigma
        # Use vmap to process each row of (N, 2) array
        # Each call to model gets a (2,) array where inputs[0]=rho, inputs[1]=sigma
        pred = jax.vmap(model)(descriptors)  # Shape (N,)
        # Subtract 1 because networks output 1 + enhancement
        pred = pred - 1.0
        return jnp.mean((pred - ref_F)**2)

### Architecture A: Shallow Network (depth=2, nodes=8)

In [7]:
# Create shallow networks
DEPTH_A = 2
NODES_A = 8
SEED = 42

# Exchange network
xnet_A = xce.net.GGA_FxNet_sigma(depth=DEPTH_A, nodes=NODES_A, seed=SEED)
print(f"Exchange Network A: depth={DEPTH_A}, nodes={NODES_A}")

# Correlation network
cnet_A = xce.net.GGA_FcNet_sigma(depth=DEPTH_A, nodes=NODES_A, seed=SEED)
print(f"Correlation Network A: depth={DEPTH_A}, nodes={NODES_A}")

Exchange Network A: depth=2, nodes=8
Correlation Network A: depth=2, nodes=8


In [8]:
# Pre-train Architecture A
print("Pre-training Architecture A...")

PRETRAIN_STEPS = 100  # Reduced for debugging
PRETRAIN_LR = 1e-2

# Learning rate schedule with decay
scheduler = optax.exponential_decay(
    init_value=PRETRAIN_LR,
    transition_begin=50,
    transition_steps=200,
    decay_rate=0.9
)
optimizer = optax.adam(learning_rate=scheduler)
loss_fn = PretrainLoss()

# Train exchange network
print("\n--- Training Exchange Network A ---")
os.chdir(CHECKPOINT_DIRS['pretrain_xnet_A'])  # Change to checkpoint dir for xcTrainer saves
trainer_x = xce.train.xcTrainer(
    model=xnet_A,
    optim=optimizer,
    steps=PRETRAIN_STEPS,
    loss=loss_fn,
    do_jit=True
)

with jax.default_device(CPU_DEVICE):
    xnet_A_trained, losses_xnet_A = trainer_x(1, xnet_A, [train_desc_x], [train_Fx])

# Save final trained model
eqx.tree_serialise_leaves('xnet_A_final.eqx', xnet_A_trained)
print(f"Saved xnet_A to {CHECKPOINT_DIRS['pretrain_xnet_A']}/xnet_A_final.eqx")
os.chdir('../..')  # Return to notebooks dir

# Train correlation network
print("\n--- Training Correlation Network A ---")
os.chdir(CHECKPOINT_DIRS['pretrain_cnet_A'])
trainer_c = xce.train.xcTrainer(
    model=cnet_A,
    optim=optimizer,
    steps=PRETRAIN_STEPS,
    loss=loss_fn,
    do_jit=True
)

with jax.default_device(CPU_DEVICE):
    cnet_A_trained, losses_cnet_A = trainer_c(1, cnet_A, [train_desc_c], [train_Fc])

# Save final trained model
eqx.tree_serialise_leaves('cnet_A_final.eqx', cnet_A_trained)
print(f"Saved cnet_A to {CHECKPOINT_DIRS['pretrain_cnet_A']}/cnet_A_final.eqx")
os.chdir('../..')

Pre-training Architecture A...

--- Training Exchange Network A ---
Epoch 0
Step = 0: initializing inp_model and inp_opt_state.
Epoch 0 :: Batch 1/1
Batch Loss = 0.3880418650418882
0, epoch_train_loss=0.3880418650418882
Epoch 1
Epoch 1 :: Batch 1/1
Batch Loss = 0.3161386213779526
1, epoch_train_loss=0.3161386213779526
Epoch 2
Epoch 2 :: Batch 1/1
Batch Loss = 0.23248599026655853
2, epoch_train_loss=0.23248599026655853
Epoch 3
Epoch 3 :: Batch 1/1
Batch Loss = 0.13019995041325894
3, epoch_train_loss=0.13019995041325894
Epoch 4
Epoch 4 :: Batch 1/1
Batch Loss = 0.05412565473210894
4, epoch_train_loss=0.05412565473210894
Epoch 5
Epoch 5 :: Batch 1/1
Batch Loss = 0.028934468091513236
5, epoch_train_loss=0.028934468091513236
Epoch 6
Epoch 6 :: Batch 1/1
Batch Loss = 0.01883311323553293
6, epoch_train_loss=0.01883311323553293
Epoch 7
Epoch 7 :: Batch 1/1
Batch Loss = 0.013155118555573523
7, epoch_train_loss=0.013155118555573523
Epoch 8
Epoch 8 :: Batch 1/1
Batch Loss = 0.009423097733914153
8

Exception ignored in: <function _xla_gc_callback at 0x7fc97c5291c0>
Traceback (most recent call last):
  File "/home/awills/anaconda3/envs/xcq/lib/python3.12/site-packages/jax/_src/lib/__init__.py", line 120, in _xla_gc_callback
    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 


Batch Loss = 0.0002620693934678523
58, epoch_train_loss=0.0002620693934678523
Epoch 59
Epoch 59 :: Batch 1/1


KeyboardInterrupt: 

### Architecture B: Deeper Network (depth=3, nodes=16)

In [ ]:
# Create deeper networks
DEPTH_B = 3
NODES_B = 16

# Exchange network
xnet_B = xce.net.GGA_FxNet_sigma(depth=DEPTH_B, nodes=NODES_B, seed=SEED)
print(f"Exchange Network B: depth={DEPTH_B}, nodes={NODES_B}")

# Correlation network  
cnet_B = xce.net.GGA_FcNet_sigma(depth=DEPTH_B, nodes=NODES_B, seed=SEED)
print(f"Correlation Network B: depth={DEPTH_B}, nodes={NODES_B}")

In [ ]:
# Pre-train Architecture B
print("Pre-training Architecture B...")

# Train exchange network
print("\n--- Training Exchange Network B ---")
os.chdir(CHECKPOINT_DIRS['pretrain_xnet_B'])
trainer_x_B = xce.train.xcTrainer(
    model=xnet_B,
    optim=optimizer,
    steps=PRETRAIN_STEPS,
    loss=loss_fn,
    do_jit=True
)

with jax.default_device(CPU_DEVICE):
    xnet_B_trained, losses_xnet_B = trainer_x_B(1, xnet_B, [train_desc_x], [train_Fx])

# Save final trained model
eqx.tree_serialise_leaves('xnet_B_final.eqx', xnet_B_trained)
print(f"Saved xnet_B to {CHECKPOINT_DIRS['pretrain_xnet_B']}/xnet_B_final.eqx")
os.chdir('../..')

# Train correlation network
print("\n--- Training Correlation Network B ---")
os.chdir(CHECKPOINT_DIRS['pretrain_cnet_B'])
trainer_c_B = xce.train.xcTrainer(
    model=cnet_B,
    optim=optimizer,
    steps=PRETRAIN_STEPS,
    loss=loss_fn,
    do_jit=True
)

with jax.default_device(CPU_DEVICE):
    cnet_B_trained, losses_cnet_B = trainer_c_B(1, cnet_B, [train_desc_c], [train_Fc])

# Save final trained model
eqx.tree_serialise_leaves('cnet_B_final.eqx', cnet_B_trained)
print(f"Saved cnet_B to {CHECKPOINT_DIRS['pretrain_cnet_B']}/cnet_B_final.eqx")
os.chdir('../..')

### Architecture C: Network with Self-Attention (depth=3, nodes=16)

This architecture uses the same depth and width as Architecture B, but includes self-attention layers to learn feature interactions dynamically.

In [ ]:
# Create networks with self-attention (same depth/nodes as Architecture B)
DEPTH_C = 3
NODES_C = 16

# Exchange network with self-attention
xnet_C = xce.net.GGA_FxNet_sigma(depth=DEPTH_C, nodes=NODES_C, seed=SEED, use_self_attention=True)
print(f"Exchange Network C: depth={DEPTH_C}, nodes={NODES_C}, self_attention=True")

# Correlation network with self-attention
cnet_C = xce.net.GGA_FcNet_sigma(depth=DEPTH_C, nodes=NODES_C, seed=SEED, use_self_attention=True)
print(f"Correlation Network C: depth={DEPTH_C}, nodes={NODES_C}, self_attention=True")

In [ ]:
# Pre-train Architecture C (with self-attention)
print("Pre-training Architecture C (with self-attention)...")

# Train exchange network with self-attention
print("\n--- Training Exchange Network C (Self-Attention) ---")
os.chdir(CHECKPOINT_DIRS['pretrain_xnet_C'])
trainer_x_C = xce.train.xcTrainer(
    model=xnet_C,
    optim=optimizer,
    steps=PRETRAIN_STEPS,
    loss=loss_fn,
    do_jit=True
)

with jax.default_device(CPU_DEVICE):
    xnet_C_trained, losses_xnet_C = trainer_x_C(1, xnet_C, [train_desc_x], [train_Fx])

# Save final trained model
eqx.tree_serialise_leaves('xnet_C_final.eqx', xnet_C_trained)
print(f"Saved xnet_C to {CHECKPOINT_DIRS['pretrain_xnet_C']}/xnet_C_final.eqx")
os.chdir('../..')

# Train correlation network with self-attention
print("\n--- Training Correlation Network C (Self-Attention) ---")
os.chdir(CHECKPOINT_DIRS['pretrain_cnet_C'])
trainer_c_C = xce.train.xcTrainer(
    model=cnet_C,
    optim=optimizer,
    steps=PRETRAIN_STEPS,
    loss=loss_fn,
    do_jit=True
)

with jax.default_device(CPU_DEVICE):
    cnet_C_trained, losses_cnet_C = trainer_c_C(1, cnet_C, [train_desc_c], [train_Fc])

# Save final trained model
eqx.tree_serialise_leaves('cnet_C_final.eqx', cnet_C_trained)
print(f"Saved cnet_C to {CHECKPOINT_DIRS['pretrain_cnet_C']}/cnet_C_final.eqx")
os.chdir('../..')

### Architecture D: Network with Log-Transformed Inputs (depth=3, nodes=16)

This architecture uses log-transformed input descriptors for improved numerical stability and training dynamics. The transformations are:
- For exchange: `x1 = (1 - exp(-s²)) * log(s + 1)` where s is the reduced density gradient
- For correlation: `x0 = log(rho^(1/3) + 1e-5)`, `x1 = (1 - exp(-s²)) * log(s + 1)`

In [ ]:
# Create networks with log-transformed inputs (same depth/nodes as Architecture B)
DEPTH_D = 3
NODES_D = 16

# Exchange network with transforms
xnet_D = xce.net.GGA_FxNet_sigma_transform(depth=DEPTH_D, nodes=NODES_D, seed=SEED, use_self_attention=False)
print(f"Exchange Network D: depth={DEPTH_D}, nodes={NODES_D}, transform=True")

# Correlation network with transforms
cnet_D = xce.net.GGA_FcNet_sigma_transform(depth=DEPTH_D, nodes=NODES_D, seed=SEED, use_self_attention=False)
print(f"Correlation Network D: depth={DEPTH_D}, nodes={NODES_D}, transform=True")

In [ ]:
# Pre-train Architecture D (with log-transforms)
print("Pre-training Architecture D (with log-transforms)...")

# Train exchange network with transforms
print("\n--- Training Exchange Network D (Transform) ---")
os.chdir(CHECKPOINT_DIRS['pretrain_xnet_D'])
trainer_x_D = xce.train.xcTrainer(
    model=xnet_D,
    optim=optimizer,
    steps=PRETRAIN_STEPS,
    loss=loss_fn,
    do_jit=True
)

with jax.default_device(CPU_DEVICE):
    xnet_D_trained, losses_xnet_D = trainer_x_D(1, xnet_D, [train_desc_x], [train_Fx])

# Save final trained model
eqx.tree_serialise_leaves('xnet_D_final.eqx', xnet_D_trained)
print(f"Saved xnet_D to {CHECKPOINT_DIRS['pretrain_xnet_D']}/xnet_D_final.eqx")
os.chdir('../..')  # Return to notebooks dir

# Train correlation network with transforms
print("\n--- Training Correlation Network D (Transform) ---")
os.chdir(CHECKPOINT_DIRS['pretrain_cnet_D'])
trainer_c_D = xce.train.xcTrainer(
    model=cnet_D,
    optim=optimizer,
    steps=PRETRAIN_STEPS,
    loss=loss_fn,
    do_jit=True
)

with jax.default_device(CPU_DEVICE):
    cnet_D_trained, losses_cnet_D = trainer_c_D(1, cnet_D, [train_desc_c], [train_Fc])

# Save final trained model
eqx.tree_serialise_leaves('cnet_D_final.eqx', cnet_D_trained)
print(f"Saved cnet_D to {CHECKPOINT_DIRS['pretrain_cnet_D']}/cnet_D_final.eqx")
os.chdir('../..')

### Architecture E: Network with Log-Transforms + Self-Attention (depth=3, nodes=16)

This architecture combines the log-transformed inputs from Architecture D with the self-attention mechanism from Architecture C.

In [ ]:
# Create networks with log-transforms AND self-attention
DEPTH_E = 3
NODES_E = 16

# Exchange network with transforms + self-attention
xnet_E = xce.net.GGA_FxNet_sigma_transform(depth=DEPTH_E, nodes=NODES_E, seed=SEED, use_self_attention=True)
print(f"Exchange Network E: depth={DEPTH_E}, nodes={NODES_E}, transform=True, self_attention=True")

# Correlation network with transforms + self-attention
cnet_E = xce.net.GGA_FcNet_sigma_transform(depth=DEPTH_E, nodes=NODES_E, seed=SEED, use_self_attention=True)
print(f"Correlation Network E: depth={DEPTH_E}, nodes={NODES_E}, transform=True, self_attention=True")

In [ ]:
# Pre-train Architecture E (with log-transforms + self-attention)
print("Pre-training Architecture E (with log-transforms + self-attention)...")

# Train exchange network
print("\n--- Training Exchange Network E (Transform + Attention) ---")
os.chdir(CHECKPOINT_DIRS['pretrain_xnet_E'])
trainer_x_E = xce.train.xcTrainer(
    model=xnet_E,
    optim=optimizer,
    steps=PRETRAIN_STEPS,
    loss=loss_fn,
    do_jit=True
)

with jax.default_device(CPU_DEVICE):
    xnet_E_trained, losses_xnet_E = trainer_x_E(1, xnet_E, [train_desc_x], [train_Fx])

# Save final trained model
eqx.tree_serialise_leaves('xnet_E_final.eqx', xnet_E_trained)
print(f"Saved xnet_E to {CHECKPOINT_DIRS['pretrain_xnet_E']}/xnet_E_final.eqx")
os.chdir('../..')  # Return to notebooks dir

# Train correlation network
print("\n--- Training Correlation Network E (Transform + Attention) ---")
os.chdir(CHECKPOINT_DIRS['pretrain_cnet_E'])
trainer_c_E = xce.train.xcTrainer(
    model=cnet_E,
    optim=optimizer,
    steps=PRETRAIN_STEPS,
    loss=loss_fn,
    do_jit=True
)

with jax.default_device(CPU_DEVICE):
    cnet_E_trained, losses_cnet_E = trainer_c_E(1, cnet_E, [train_desc_c], [train_Fc])

# Save final trained model
eqx.tree_serialise_leaves('cnet_E_final.eqx', cnet_E_trained)
print(f"Saved cnet_E to {CHECKPOINT_DIRS['pretrain_cnet_E']}/cnet_E_final.eqx")
os.chdir('../..')

In [ ]:
# Compare pre-training results
print("\n" + "="*60)
print("Pre-training Results Summary")
print("="*60)

# Compute RMSE for each architecture
def compute_rmse(model, desc, ref):
    pred = jax.vmap(model)(desc)
    return jnp.sqrt(jnp.mean((pred - ref)**2))

# Exchange networks
rmse_Fx_A = compute_rmse(xnet_A_trained, train_desc_x, train_Fx)
rmse_Fx_B = compute_rmse(xnet_B_trained, train_desc_x, train_Fx)
rmse_Fx_C = compute_rmse(xnet_C_trained, train_desc_x, train_Fx)
rmse_Fx_D = compute_rmse(xnet_D_trained, train_desc_x, train_Fx)
rmse_Fx_E = compute_rmse(xnet_E_trained, train_desc_x, train_Fx)

# Correlation networks
rmse_Fc_A = compute_rmse(cnet_A_trained, train_desc_c, train_Fc)
rmse_Fc_B = compute_rmse(cnet_B_trained, train_desc_c, train_Fc)
rmse_Fc_C = compute_rmse(cnet_C_trained, train_desc_c, train_Fc)
rmse_Fc_D = compute_rmse(cnet_D_trained, train_desc_c, train_Fc)
rmse_Fc_E = compute_rmse(cnet_E_trained, train_desc_c, train_Fc)

print("\nExchange Enhancement Factor (Fx) RMSE:")
print(f"  Architecture A (depth={DEPTH_A}, nodes={NODES_A}):                          {rmse_Fx_A:.6f}")
print(f"  Architecture B (depth={DEPTH_B}, nodes={NODES_B}):                         {rmse_Fx_B:.6f}")
print(f"  Architecture C (depth={DEPTH_C}, nodes={NODES_C}, attention):              {rmse_Fx_C:.6f}")
print(f"  Architecture D (depth={DEPTH_D}, nodes={NODES_D}, transform):              {rmse_Fx_D:.6f}")
print(f"  Architecture E (depth={DEPTH_E}, nodes={NODES_E}, transform+attention):    {rmse_Fx_E:.6f}")

print("\nCorrelation Enhancement Factor (Fc) RMSE:")
print(f"  Architecture A (depth={DEPTH_A}, nodes={NODES_A}):                          {rmse_Fc_A:.6f}")
print(f"  Architecture B (depth={DEPTH_B}, nodes={NODES_B}):                         {rmse_Fc_B:.6f}")
print(f"  Architecture C (depth={DEPTH_C}, nodes={NODES_C}, attention):              {rmse_Fc_C:.6f}")
print(f"  Architecture D (depth={DEPTH_D}, nodes={NODES_D}, transform):              {rmse_Fc_D:.6f}")
print(f"  Architecture E (depth={DEPTH_E}, nodes={NODES_E}, transform+attention):    {rmse_Fc_E:.6f}")


In [ ]:
# Visualize pre-training results
fig, axes = plt.subplots(2, 5, figsize=(25, 10))

# Exchange - Architecture A
ax = axes[0, 0]
pred_A = jax.vmap(xnet_A_trained)(train_desc_x)
ax.scatter(train_Fx, pred_A, alpha=0.5, s=1)
ax.plot([train_Fx.min(), train_Fx.max()], [train_Fx.min(), train_Fx.max()], 'r--')
ax.set_xlabel('Reference Fx')
ax.set_ylabel('Predicted Fx')
ax.set_title(f'Exchange - Arch A (RMSE={rmse_Fx_A:.4f})')

# Exchange - Architecture B
ax = axes[0, 1]
pred_B = jax.vmap(xnet_B_trained)(train_desc_x)
ax.scatter(train_Fx, pred_B, alpha=0.5, s=1)
ax.plot([train_Fx.min(), train_Fx.max()], [train_Fx.min(), train_Fx.max()], 'r--')
ax.set_xlabel('Reference Fx')
ax.set_ylabel('Predicted Fx')
ax.set_title(f'Exchange - Arch B (RMSE={rmse_Fx_B:.4f})')

# Exchange - Architecture C (Self-Attention)
ax = axes[0, 2]
pred_C = jax.vmap(xnet_C_trained)(train_desc_x)
ax.scatter(train_Fx, pred_C, alpha=0.5, s=1)
ax.plot([train_Fx.min(), train_Fx.max()], [train_Fx.min(), train_Fx.max()], 'r--')
ax.set_xlabel('Reference Fx')
ax.set_ylabel('Predicted Fx')
ax.set_title(f'Exchange - Arch C Attn (RMSE={rmse_Fx_C:.4f})')

# Exchange - Architecture D (Transform)
ax = axes[0, 3]
pred_D = jax.vmap(xnet_D_trained)(train_desc_x)
ax.scatter(train_Fx, pred_D, alpha=0.5, s=1)
ax.plot([train_Fx.min(), train_Fx.max()], [train_Fx.min(), train_Fx.max()], 'r--')
ax.set_xlabel('Reference Fx')
ax.set_ylabel('Predicted Fx')
ax.set_title(f'Exchange - Arch D Transform (RMSE={rmse_Fx_D:.4f})')

# Exchange - Architecture E (Transform + Attention)
ax = axes[0, 4]
pred_E = jax.vmap(xnet_E_trained)(train_desc_x)
ax.scatter(train_Fx, pred_E, alpha=0.5, s=1)
ax.plot([train_Fx.min(), train_Fx.max()], [train_Fx.min(), train_Fx.max()], 'r--')
ax.set_xlabel('Reference Fx')
ax.set_ylabel('Predicted Fx')
ax.set_title(f'Exchange - Arch E Trans+Attn (RMSE={rmse_Fx_E:.4f})')

# Correlation - Architecture A
ax = axes[1, 0]
pred_A = jax.vmap(cnet_A_trained)(train_desc_c)
ax.scatter(train_Fc, pred_A, alpha=0.5, s=1)
ax.plot([train_Fc.min(), train_Fc.max()], [train_Fc.min(), train_Fc.max()], 'r--')
ax.set_xlabel('Reference Fc')
ax.set_ylabel('Predicted Fc')
ax.set_title(f'Correlation - Arch A (RMSE={rmse_Fc_A:.4f})')

# Correlation - Architecture B
ax = axes[1, 1]
pred_B = jax.vmap(cnet_B_trained)(train_desc_c)
ax.scatter(train_Fc, pred_B, alpha=0.5, s=1)
ax.plot([train_Fc.min(), train_Fc.max()], [train_Fc.min(), train_Fc.max()], 'r--')
ax.set_xlabel('Reference Fc')
ax.set_ylabel('Predicted Fc')
ax.set_title(f'Correlation - Arch B (RMSE={rmse_Fc_B:.4f})')

# Correlation - Architecture C (Self-Attention)
ax = axes[1, 2]
pred_C = jax.vmap(cnet_C_trained)(train_desc_c)
ax.scatter(train_Fc, pred_C, alpha=0.5, s=1)
ax.plot([train_Fc.min(), train_Fc.max()], [train_Fc.min(), train_Fc.max()], 'r--')
ax.set_xlabel('Reference Fc')
ax.set_ylabel('Predicted Fc')
ax.set_title(f'Correlation - Arch C Attn (RMSE={rmse_Fc_C:.4f})')

# Correlation - Architecture D (Transform)
ax = axes[1, 3]
pred_D = jax.vmap(cnet_D_trained)(train_desc_c)
ax.scatter(train_Fc, pred_D, alpha=0.5, s=1)
ax.plot([train_Fc.min(), train_Fc.max()], [train_Fc.min(), train_Fc.max()], 'r--')
ax.set_xlabel('Reference Fc')
ax.set_ylabel('Predicted Fc')
ax.set_title(f'Correlation - Arch D Transform (RMSE={rmse_Fc_D:.4f})')

# Correlation - Architecture E (Transform + Attention)
ax = axes[1, 4]
pred_E = jax.vmap(cnet_E_trained)(train_desc_c)
ax.scatter(train_Fc, pred_E, alpha=0.5, s=1)
ax.plot([train_Fc.min(), train_Fc.max()], [train_Fc.min(), train_Fc.max()], 'r--')
ax.set_xlabel('Reference Fc')
ax.set_ylabel('Predicted Fc')
ax.set_title(f'Correlation - Arch E Trans+Attn (RMSE={rmse_Fc_E:.4f})')

plt.tight_layout()
plt.savefig('pretrain_comparison_all_architectures.png', dpi=150)
plt.show()


---
## 3. Training Part I: Total Energies and Atomization Energies

Now we fine-tune the pre-trained networks on:
1. Total energies of atoms (compared to reference CCSD(T) or high-accuracy DFT)
2. Atomization energies of small molecules

We use PySCF-AD to enable gradient-based optimization through the SCF cycle.

In [ ]:
# Create combined XC model from pre-trained networks
# We'll use Architecture B (deeper network) for training

def lda_c_pw(rhoa, rhob):
    '''PW92 LDA correlation for spin-polarized systems.'''
    params_a_pp = [1, 1, 1]
    params_a_a = [0.031091, 0.015545, 0.016887]
    params_a_alpha1 = [0.21370, 0.20548, 0.11125]
    params_a_beta1 = [7.5957, 14.1189, 10.357]
    params_a_beta2 = [3.5876, 6.1977, 3.6231]
    params_a_beta3 = [1.6382, 3.3662, 0.88026]
    params_a_beta4 = [0.49294, 0.62517, 0.49671]
    params_a_fz20 = 1.709921

    rho = rhoa + rhob
    zeta = (rhoa - rhob) / (rhoa + rhob + 1e-8)
    rs = (4 * jnp.pi * (rho + 1e-8) / 3)**(-1/3)

    def g_aux(k, rs):
        return params_a_beta1[k] * jnp.sqrt(rs) + params_a_beta2[k] * rs \
            + params_a_beta3[k] * rs**1.5 + params_a_beta4[k] * rs**(params_a_pp[k] + 1)

    def g(k, rs):
        return -2 * params_a_a[k] * (1 + params_a_alpha1[k] * rs) \
            * jnp.log(1 + 1 / (2 * params_a_a[k] * g_aux(k, rs)))

    def f_zeta(zeta):
        return ((1 + zeta)**(4/3) + (1 - zeta)**(4/3) - 2) / (2**(4/3) - 2)

    def f_pw(rs, zeta):
        return g(0, rs) + zeta**4 * f_zeta(zeta) * (g(1, rs) - g(0, rs) + g(2, rs) / params_a_fz20) \
            - f_zeta(zeta) * g(2, rs) / params_a_fz20

    return f_pw(rs, zeta)


class RXCModel_GGA(eqx.Module):
    '''
    Combined GGA XC model with exchange and correlation networks.
    
    Computes epsilon = rho * e_xc where e_xc = ex_lda * Fx + ec_lda * Fc.
    Works with both unpolarized and spin-polarized inputs.
    
    Input format: (N, D) where D=2 for unpolarized, D=5 for polarized, N=grid points
    Each row is one grid point: [rho, sigma] or [rho_a, rho_b, sigma_aa, sigma_ab, sigma_bb]
    
    The networks expect one grid point at a time, so vmap is used internally.
    '''
    xnet: eqx.Module
    cnet: eqx.Module

    def __init__(self, xnet, cnet):
        self.xnet = xnet
        self.cnet = cnet
    
    def _eval_single_point_unpol(self, point):
        '''Evaluate epsilon for a single unpolarized grid point.'''
        rho = point[0]
        sigma = point[1]
        
        ex_lda = lda_x(rho)
        ec_pw92 = pw92c_unpolarized(rho)
        
        rho_safe = jnp.maximum(rho, 1e-18)
        # Networks expect [rho, sigma] as input
        Fx = self.xnet(point)
        Fc = self.cnet(point)
        epsilon = rho_safe * (ex_lda * Fx + ec_pw92 * Fc)
        return epsilon
    
    def _eval_single_point_pol(self, point):
        '''Evaluate epsilon for a single polarized grid point.'''
        rho_a = point[0]
        rho_b = point[1]
        sigma_aa = point[2]
        sigma_ab = point[3]
        sigma_bb = point[4]
        
        rho = rho_a + rho_b
        sigma = sigma_aa + 2*sigma_ab + sigma_bb
        
        ex_lda = lda_x(rho)
        ec_pw92 = lda_c_pw(rho_a, rho_b)
        
        rho_safe = jnp.maximum(rho, 1e-18)
        # Networks expect [rho, sigma] - combine for network input
        net_input = jnp.array([rho, sigma])
        Fx = self.xnet(net_input)
        Fc = self.cnet(net_input)
        epsilon = rho_safe * (ex_lda * Fx + ec_pw92 * Fc)
        return epsilon
        
    def __call__(self, inputs):
        '''
        Compute epsilon = rho * e_xc.
        
        :param inputs: Shape (N, 2) for unpolarized or (N, 5) for polarized
                       Each row is one grid point
        :return: epsilon array of shape (N,)
        '''
        inputs = jnp.atleast_2d(inputs)
        
        # Check second dimension to determine polarization
        if inputs.shape[-1] == 2:
            # Unpolarized case: inputs shape (N, 2) with [rho, sigma] per row
            epsilon = jax.vmap(self._eval_single_point_unpol)(inputs)
            
        elif inputs.shape[-1] == 5:
            # Polarized case: inputs shape (N, 5) 
            epsilon = jax.vmap(self._eval_single_point_pol)(inputs)
        else:
            raise ValueError(f"Unexpected input shape: {inputs.shape}. Expected (N, 2) or (N, 5)")
        
        return jnp.squeeze(epsilon)


# Create the XC model from pre-trained networks
xcmodel = RXCModel_GGA(xnet=xnet_B_trained, cnet=cnet_B_trained)
print("Created combined XC model from pre-trained Architecture B networks")

In [ ]:
# Training data: atoms and small molecules with reference energies
# Use pyscf-ad (gto_ad) for autodifferentiable molecules

# Define systems with atom specifications
training_systems = [
    # Atoms
    {'name': 'H', 'atoms': 'H 0 0 0', 'spin': 1, 'charge': 0, 'type': 'atom'},
    {'name': 'O', 'atoms': 'O 0 0 0', 'spin': 2, 'charge': 0, 'type': 'atom'},
    # Molecules
    {'name': 'H2O', 'atoms': 'O 0 0 0; H 0 0.757 0.587; H 0 -0.757 0.587', 
     'spin': 0, 'charge': 0, 'type': 'molecule', 'composition': {'H': 2, 'O': 1}},
]

# Reference energies (Hartree)
# H: -0.5 Ha (exact)
# O: -75.0673 Ha (high-level reference)
# H2O atomization: ~974.94 kJ/mol = ~0.371 Ha
kjMol_to_H = 2625.5
refs = {
    'H_TE': -0.5,
    'O_TE': -75.0673,
    'H2O_AE': -974.94 / kjMol_to_H  # Atomization energy (negative = exothermic)
}

# Build pyscf-ad molecules (autodifferentiable)
print("Building pyscf-ad molecules for training...")
mols_ad = []
mol_names = []

for system in training_systems:
    mol = gto_ad.Mole()
    mol.atom = system['atoms']
    mol.basis = BASIS
    mol.charge = system['charge']
    mol.spin = system['spin']
    mol.build()
    mol.max_memory = 32000
    mols_ad.append(mol)
    mol_names.append(system['name'])
    print(f"  Built {system['name']}: {mol.nelectron} electrons, spin={system['spin']}")

print(f"\nTotal {len(mols_ad)} molecules for training")
print(f"Reference energies: {refs}")

In [ ]:
# Define the custom eval_xc_gga_j2 function from gga_pol_dev.ipynb
# This handles BOTH polarized and unpolarized cases properly
# Uses vmap internally for per-grid-point network evaluation

def eval_xc_gga_j2(xc_code, rho, spin=0, relativity=0, deriv=1, omega=None, verbose=None,
                   xcmodel=None):
    '''
    Evaluate GGA exchange-correlation using a neural network model.
    
    Handles both spin-polarized and unpolarized cases.
    Uses vmap internally for per-grid-point evaluation.
    
    :param xc_code: XC functional code (ignored)
    :param rho: Density and gradients from PySCF
    :param spin: Spin polarization flag
    :param xcmodel: Combined XC model (RXCModel_GGA)
    :return: (exc, vxc, fxc, kxc)
    '''
    # Detect if spin-polarized by checking if rho is a tuple/list
    try:
        # Try unpolarized first
        rho0, dx, dy, dz = rho[:4]
        sigma = jnp.array(dx**2 + dy**2 + dz**2)
        rho0 = jnp.array(rho0)
        is_polarized = False
    except (ValueError, TypeError):
        # Spin-polarized: rho = [rho_a, rho_b]
        rho_a, rho_b = rho
        rho0a, dxa, dya, dza = rho_a[:4]
        rho0b, dxb, dyb, dzb = rho_b[:4]
        
        rho0 = rho0a + rho0b
        sigma_aa = dxa**2 + dya**2 + dza**2
        sigma_ab = dxa*dxb + dya*dyb + dza*dzb  
        sigma_bb = dxb**2 + dyb**2 + dzb**2
        is_polarized = True
    
    if not is_polarized:
        # ============ UNPOLARIZED CASE ============
        # Stack as (N, 2) - each row is one grid point [rho, sigma]
        rhosig = jnp.stack([rho0, sigma], axis=1)  # Shape (N, 2)
        
        # Compute exc = epsilon / rho (model uses vmap internally)
        epsilon = xcmodel(rhosig)
        exc = epsilon / (rho0 + 1e-18)
        
        # First derivatives using vmap over grid points
        # Each point is (2,) with [rho, sigma]
        def single_point_model(point):
            return xcmodel(point.reshape(1, 2)).squeeze()
        
        vrho_f = eqx.filter_grad(single_point_model)
        v1 = jnp.array(jax.vmap(vrho_f)(rhosig))  # Shape (N, 2)
        vrho = v1[:, 0]
        vsigma = v1[:, 1]
        vxc = (vrho, vsigma, None, None)
        
        # Second derivatives (Hessian)
        v2_f = jax.hessian(single_point_model)
        v2 = jnp.array(jax.vmap(v2_f)(rhosig))  # Shape (N, 2, 2)
        
        v2rho2 = v2[:, 0, 0]
        v2rhosigma = v2[:, 0, 1]
        v2sigma2 = v2[:, 1, 1]
        
        fxc = (v2rho2, v2rhosigma, v2sigma2, 
               None, None, None, None, None, None, None)
        kxc = None
        
    else:
        # ============ POLARIZED CASE ============
        # Stack as (N, 5) - each row is one grid point
        rhosig = jnp.stack([rho0a, rho0b, sigma_aa, sigma_ab, sigma_bb], axis=1)  # Shape (N, 5)
        
        # Compute exc (model uses vmap internally)
        epsilon = xcmodel(rhosig)
        exc = epsilon / (rho0 + 1e-18)
        
        # First derivatives
        def single_point_model_pol(point):
            return xcmodel(point.reshape(1, 5)).squeeze()
        
        vrho_f = eqx.filter_grad(single_point_model_pol)
        v1 = jnp.array(jax.vmap(vrho_f)(rhosig))  # Shape (N, 5)
        
        # vrho = [vrho_a, vrho_b]
        vrho = jnp.stack([v1[:, 0], v1[:, 1]], axis=1)
        # vsigma = [vsigma_aa, vsigma_ab, vsigma_bb]
        vsigma = jnp.stack([v1[:, 2], v1[:, 3], v1[:, 4]], axis=1)
        vxc = (vrho, vsigma, None, None)
        
        # Second derivatives
        v2_f = jax.hessian(single_point_model_pol)
        v2 = jnp.array(jax.vmap(v2_f)(rhosig))  # Shape (N, 5, 5)
        
        # v2rho2 = [aa, ab, bb]
        v2rho2 = jnp.stack([v2[:, 0, 0], v2[:, 0, 1], v2[:, 1, 1]], axis=1)
        
        # v2rhosigma = [a-aa, a-ab, a-bb, b-aa, b-ab, b-bb]
        v2rhosigma = jnp.stack([
            v2[:, 0, 2], v2[:, 0, 3], v2[:, 0, 4],
            v2[:, 1, 2], v2[:, 1, 3], v2[:, 1, 4]
        ], axis=1)
        
        # v2sigma2 = [aa-aa, aa-ab, aa-bb, ab-ab, ab-bb, bb-bb]
        v2sigma2 = jnp.stack([
            v2[:, 2, 2], v2[:, 2, 3], v2[:, 2, 4],
            v2[:, 3, 3], v2[:, 3, 4], v2[:, 4, 4]
        ], axis=1)
        
        fxc = (v2rho2, v2rhosigma, v2sigma2,
               None, None, None, None, None, None, None)
        kxc = None
    
    return exc, vxc, fxc, kxc

print("Defined custom eval_xc_gga_j2 (from gga_pol_dev.ipynb)")
print("This handles both polarized and unpolarized cases")
print("Input format: (N, D) where D=2 (unpolarized) or D=5 (polarized)")

In [ ]:
# Define the loss function that runs SCF through pyscf-ad
# This is the key: mf.kernel() runs the actual SCF cycle with autodiff

@eqx.filter_value_and_grad
def energy_loss(model, mols, refs):
    '''
    Loss function for training on total energies and atomization energies.
    
    Runs actual SCF calculations through pyscf-ad for automatic differentiation.
    Uses eval_xc_gga_j2 from xcquinox.pyscf as the XC evaluator.
    
    :param model: XC model (RXCModel_GGA)
    :param mols: List of pyscf-ad Mole objects [H, O, H2O]
    :param refs: Dict of reference energies
    :return: Total loss value
    '''
    preds = []
    
    for idx, mol in enumerate(mols):
        # Select RKS or UKS based on spin
        if mol.spin:
            mf = dft_ad.UKS(mol)
        else:
            mf = dft_ad.RKS(mol)
        
        # Configure calculation for stability during training
        mf.grids.level = GRID_LEVEL  # Use low grid for speed
        mf.diis = False  # DIIS can cause issues with autodiff
        mf.damp = 0.5    # Damping for stability
        mf.max_cycle = 25
        
        # Set custom XC functional using eval_xc_gga_j2 from xcquinox.pyscf
        custom_eval_xc = partial(eval_xc_gga_j2, xcmodel=model)
        mf.define_xc_(custom_eval_xc, 'GGA')
        
        # Run SCF - this is where gradients flow through!
        pred = mf.kernel()
        preds.append(pred)
        
        jax.debug.print("Mol {idx}: E = {pred}", idx=idx, pred=pred)
    
    E_H, E_O, E_H2O = preds
    
    # Compute atomization energy: AE = E(H2O) - 2*E(H) - E(O)
    pred_AE = E_H2O - 2*E_H - E_O
    ref_AE = refs['H2O_AE']
    
    # Loss components
    AE_loss = jnp.sqrt((pred_AE - ref_AE)**2)
    H_loss = jnp.sqrt((E_H - refs['H_TE'])**2)
    O_loss = jnp.sqrt((E_O - refs['O_TE'])**2)
    
    # Combined loss with atomization energy weighted heavily
    total_loss = 50.0 * AE_loss + H_loss + O_loss
    
    jax.debug.print("AE: pred={pred}, ref={ref}, error={err}", 
                   pred=pred_AE, ref=ref_AE, err=AE_loss)
    jax.debug.print("Total loss: {loss}", loss=total_loss)
    
    return total_loss

print("Defined energy loss function with pyscf-ad SCF using eval_xc_gga_j2")

In [ ]:
# Training on Total Energies using PySCF-AD SCF
# This properly backpropagates through the SCF cycle

print("\n" + "="*60)
print("Training on Total Energies (PySCF-AD SCF)")
print("="*60)

# Training configuration
TRAIN_STEPS = 25  # Reduced for debugging
TRAIN_LR_INIT = 5e-3
TRAIN_LR_END = 1e-5
DECAY_BEGIN = 12

# Learning rate schedule
lr_scheduler = optax.linear_schedule(
    init_value=TRAIN_LR_INIT,
    transition_steps=TRAIN_STEPS - DECAY_BEGIN,
    transition_begin=DECAY_BEGIN,
    end_value=TRAIN_LR_END,
)

# Optimizer with gradient clipping for stability
train_optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adam(learning_rate=lr_scheduler)
)

# Use the xcquinox Optimizer class
print(f"\nTraining for {TRAIN_STEPS} steps with pyscf-ad...")
print(f"Initial LR: {TRAIN_LR_INIT}, Final LR: {TRAIN_LR_END}")

energy_trainer = Optimizer(
    model=xcmodel,
    optim=train_optimizer,
    mols=mols_ad,
    refs=refs,
    loss=energy_loss,
    print_every=5,
    steps=TRAIN_STEPS
)

# Run training - this runs actual SCF through pyscf-ad!
xcmodel_trained, training_losses = energy_trainer()

print(f"\nTraining complete!")
print(f"Final loss: {training_losses[-1]:.6f}")

# Save checkpoint
energy_ckpt_dir = CHECKPOINT_DIRS['train_energy']
eqx.tree_serialise_leaves(os.path.join(energy_ckpt_dir, 'xcmodel_energy.eqx'), xcmodel_trained)
print(f"Saved trained model to {energy_ckpt_dir}/xcmodel_energy.eqx")

### Energy-Only Training with Self-Attention Networks

Now we train the self-attention architecture (Architecture C) using the same energy-only loss function.

In [ ]:
# Create XC model with self-attention networks (Architecture C)
xcmodel_attn = RXCModel_GGA(xnet=xnet_C_trained, cnet=cnet_C_trained)
print("Created combined XC model from pre-trained Architecture C networks (with self-attention)")

In [ ]:
# Training on Total Energies with Self-Attention Networks
print("\n" + "="*60)
print("Training on Total Energies (Self-Attention Networks)")
print("="*60)

# Use same optimizer and training configuration
train_optimizer_attn = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adam(learning_rate=lr_scheduler)
)

print(f"\nTraining for {TRAIN_STEPS} steps with pyscf-ad (self-attention model)...")

energy_trainer_attn = Optimizer(
    model=xcmodel_attn,
    optim=train_optimizer_attn,
    mols=mols_ad,
    refs=refs,
    loss=energy_loss,
    print_every=5,
    steps=TRAIN_STEPS
)

# Run training
xcmodel_attn_trained, training_losses_attn = energy_trainer_attn()

print(f"\nTraining complete!")
print(f"Final loss: {training_losses_attn[-1]:.6f}")

# Save checkpoint
energy_attn_ckpt_dir = CHECKPOINT_DIRS['train_energy_attn']
eqx.tree_serialise_leaves(os.path.join(energy_attn_ckpt_dir, 'xcmodel_energy_attn.eqx'), xcmodel_attn_trained)
print(f"Saved trained model to {energy_attn_ckpt_dir}/xcmodel_energy_attn.eqx")

---
## 4. Training Part II: Including Density Error

Now we add density error to the loss function. This penalizes deviations in the predicted electron density from a reference (e.g., CCSD density).

In [ ]:
# Define loss function that includes density error
# This adds a penalty for density deviations on top of energy errors

@eqx.filter_value_and_grad
def energy_density_loss(model, mols, refs, energy_weight=1.0, density_weight=0.1):
    '''
    Loss function that combines energy error and density regularization.
    
    Runs SCF through pyscf-ad using eval_xc_gga_j2 and adds a density smoothness penalty.
    
    :param model: XC model (RXCModel_GGA)
    :param mols: List of pyscf-ad Mole objects [H, O, H2O]
    :param refs: Dict of reference energies
    :param energy_weight: Weight for energy loss component
    :param density_weight: Weight for density regularization
    :return: Total loss value
    '''
    preds = []
    density_penalties = []
    
    for idx, mol in enumerate(mols):
        # Select RKS or UKS based on spin
        if mol.spin:
            mf = dft_ad.UKS(mol)
        else:
            mf = dft_ad.RKS(mol)
        
        # Configure calculation
        mf.grids.level = GRID_LEVEL
        mf.diis = False
        mf.damp = 0.5
        mf.max_cycle = 25
        
        # Set custom XC functional using eval_xc_gga_j2 from xcquinox.pyscf
        custom_eval_xc = partial(eval_xc_gga_j2, xcmodel=model)
        mf.define_xc_(custom_eval_xc, 'GGA')
        
        # Run SCF
        pred = mf.kernel()
        preds.append(pred)
        
        # Compute density penalty (gradient smoothness)
        # This encourages smooth density profiles
        dm = mf.make_rdm1()
        if dm.ndim == 3:  # UKS
            dm_total = dm[0] + dm[1]
        else:
            dm_total = dm
        
        # Simple density matrix norm as regularization
        dm_penalty = jnp.sum(dm_total**2) / mol.nelectron
        density_penalties.append(dm_penalty)
        
        jax.debug.print("Mol {idx}: E = {pred}, DM_penalty = {dmp}", 
                       idx=idx, pred=pred, dmp=dm_penalty)
    
    E_H, E_O, E_H2O = preds
    
    # Compute atomization energy
    pred_AE = E_H2O - 2*E_H - E_O
    ref_AE = refs['H2O_AE']
    
    # Energy loss components
    AE_loss = jnp.sqrt((pred_AE - ref_AE)**2)
    H_loss = jnp.sqrt((E_H - refs['H_TE'])**2)
    O_loss = jnp.sqrt((E_O - refs['O_TE'])**2)
    
    energy_loss_val = 50.0 * AE_loss + H_loss + O_loss
    
    # Density regularization (normalize by number of systems)
    density_loss_val = jnp.mean(jnp.array(density_penalties))
    
    # Combined loss
    total_loss = energy_weight * energy_loss_val + density_weight * density_loss_val
    
    jax.debug.print("Energy loss: {el}, Density loss: {dl}, Total: {tl}",
                   el=energy_loss_val, dl=density_loss_val, tl=total_loss)
    
    return total_loss

print("Defined energy+density loss function with pyscf-ad SCF using eval_xc_gga_j2")

In [ ]:
# Training with Energy + Density Error Loss using PySCF-AD SCF
print("\n" + "="*60)
print("Training with Energy + Density Error Loss (PySCF-AD)")
print("="*60)

# Reset model to pre-trained state for fair comparison
xcmodel_density = RXCModel_GGA(xnet=xnet_B_trained, cnet=cnet_B_trained)

# Training configuration
TRAIN_STEPS_2 = 25  # Reduced for debugging
ENERGY_WEIGHT = 1.0
DENSITY_WEIGHT = 0.1

# Learning rate schedule
lr_scheduler_2 = optax.linear_schedule(
    init_value=TRAIN_LR_INIT,
    transition_steps=TRAIN_STEPS_2 - DECAY_BEGIN,
    transition_begin=DECAY_BEGIN,
    end_value=TRAIN_LR_END,
)

# Optimizer with gradient clipping
train_optimizer_2 = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adam(learning_rate=lr_scheduler_2)
)

# Create a partial loss function with weights
energy_density_loss_weighted = partial(
    energy_density_loss, 
    energy_weight=ENERGY_WEIGHT, 
    density_weight=DENSITY_WEIGHT
)

print(f"\nTraining with energy_weight={ENERGY_WEIGHT}, density_weight={DENSITY_WEIGHT}")
print(f"Training for {TRAIN_STEPS_2} steps with pyscf-ad...")

density_trainer = Optimizer(
    model=xcmodel_density,
    optim=train_optimizer_2,
    mols=mols_ad,
    refs=refs,
    loss=energy_density_loss_weighted,
    print_every=5,
    steps=TRAIN_STEPS_2
)

# Run training
xcmodel_density_trained, density_losses = density_trainer()

print(f"\nTraining complete!")
print(f"Final loss: {density_losses[-1]:.6f}")

# Save checkpoint
density_ckpt_dir = CHECKPOINT_DIRS['train_energy_density']
eqx.tree_serialise_leaves(os.path.join(density_ckpt_dir, 'xcmodel_density.eqx'), xcmodel_density_trained)
print(f"Saved trained model to {density_ckpt_dir}/xcmodel_density.eqx")

In [ ]:
# Training with Energy + Density Error Loss (Higher Density Weight)
print("\n" + "="*60)
print("Training with Energy + DM Regularization (Higher Weight)")
print("="*60)

# Reset model to pre-trained state
xcmodel_density_hw = RXCModel_GGA(xnet=xnet_B_trained, cnet=cnet_B_trained)

# Training configuration - higher density weight
TRAIN_STEPS_HW = 25  # Reduced for debugging
ENERGY_WEIGHT_HW = 1.0
DENSITY_WEIGHT_HW = 10.0  # 10x higher than previous

# Learning rate schedule
lr_scheduler_hw = optax.linear_schedule(
    init_value=TRAIN_LR_INIT,
    transition_steps=TRAIN_STEPS_HW - DECAY_BEGIN,
    transition_begin=DECAY_BEGIN,
    end_value=TRAIN_LR_END,
)

# Optimizer with gradient clipping
train_optimizer_hw = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adam(learning_rate=lr_scheduler_hw)
)

# Create a partial loss function with higher density weight
energy_density_loss_hw = partial(
    energy_density_loss, 
    energy_weight=ENERGY_WEIGHT_HW, 
    density_weight=DENSITY_WEIGHT_HW
)

print(f"\nTraining with energy_weight={ENERGY_WEIGHT_HW}, density_weight={DENSITY_WEIGHT_HW}")
print(f"Training for {TRAIN_STEPS_HW} steps with pyscf-ad...")

density_trainer_hw = Optimizer(
    model=xcmodel_density_hw,
    optim=train_optimizer_hw,
    mols=mols_ad,
    refs=refs,
    loss=energy_density_loss_hw,
    print_every=5,
    steps=TRAIN_STEPS_HW
)

# Run training
xcmodel_density_trained_hw, density_losses_hw = density_trainer_hw()

print(f"\nTraining complete!")
print(f"Final loss: {density_losses_hw[-1]:.6f}")

# Save checkpoint
density_hw_ckpt_dir = CHECKPOINT_DIRS['train_energy_density_hw']
eqx.tree_serialise_leaves(os.path.join(density_hw_ckpt_dir, 'xcmodel_density_hw.eqx'), xcmodel_density_trained_hw)
print(f"Saved trained model to {density_hw_ckpt_dir}/xcmodel_density_hw.eqx")

### Energy + DM Regularization with Self-Attention Networks

Training the self-attention architecture with energy + density matrix regularization loss.

In [ ]:
# Training with Energy + DM Regularization (Self-Attention Networks)
print("\n" + "="*60)
print("Training with Energy + DM Reg (Self-Attention, weight=0.1)")
print("="*60)

# Reset model to pre-trained self-attention state
xcmodel_density_attn = RXCModel_GGA(xnet=xnet_C_trained, cnet=cnet_C_trained)

# Learning rate schedule
lr_scheduler_attn = optax.linear_schedule(
    init_value=TRAIN_LR_INIT,
    transition_steps=TRAIN_STEPS_2 - DECAY_BEGIN,
    transition_begin=DECAY_BEGIN,
    end_value=TRAIN_LR_END,
)

train_optimizer_density_attn = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adam(learning_rate=lr_scheduler_attn)
)

# Create partial loss function with weights
energy_density_loss_attn = partial(
    energy_density_loss, 
    energy_weight=ENERGY_WEIGHT, 
    density_weight=DENSITY_WEIGHT
)

print(f"\nTraining with energy_weight={ENERGY_WEIGHT}, density_weight={DENSITY_WEIGHT}")
print(f"Training for {TRAIN_STEPS_2} steps with pyscf-ad (self-attention model)...")

density_trainer_attn = Optimizer(
    model=xcmodel_density_attn,
    optim=train_optimizer_density_attn,
    mols=mols_ad,
    refs=refs,
    loss=energy_density_loss_attn,
    print_every=5,
    steps=TRAIN_STEPS_2
)

# Run training
xcmodel_density_attn_trained, density_losses_attn = density_trainer_attn()

print(f"\nTraining complete!")
print(f"Final loss: {density_losses_attn[-1]:.6f}")

# Save checkpoint
density_attn_ckpt_dir = CHECKPOINT_DIRS['train_energy_density_attn']
eqx.tree_serialise_leaves(os.path.join(density_attn_ckpt_dir, 'xcmodel_density_attn.eqx'), xcmodel_density_attn_trained)
print(f"Saved trained model to {density_attn_ckpt_dir}/xcmodel_density_attn.eqx")

### Training with Grid Density Loss

Now we train using the actual electron density evaluated on the grid, not just the density matrix norm. This compares the predicted density at each grid point to a reference density (from PBE).

In [ ]:
# Compute reference densities on grid using pyscf-ad (same as training)
# This ensures the grids match exactly

print("Computing reference densities on grid from PBE using pyscf-ad...")
ref_densities = {}

for mol, name in zip(mols_ad, mol_names):
    # Run PBE calculation with pyscf-ad
    if mol.spin:
        mf_ref = dft_ad.UKS(mol)
    else:
        mf_ref = dft_ad.RKS(mol)
    
    mf_ref.xc = REFERENCE_XC
    mf_ref.grids.level = GRID_LEVEL
    mf_ref.kernel()
    
    # Get density on grid
    ao = mf_ref._numint.eval_ao(mol, mf_ref.grids.coords, deriv=0)
    dm = mf_ref.make_rdm1()
    
    if dm.ndim == 3:  # UKS - sum alpha and beta
        dm_total = dm[0] + dm[1]
    else:
        dm_total = dm
    
    # Compute density on grid
    rho_ref = mf_ref._numint.eval_rho(mol, ao, dm_total, xctype='LDA')
    
    ref_densities[name] = {
        'rho': jnp.array(rho_ref),
        'weights': jnp.array(mf_ref.grids.weights),
        'ngrids': len(rho_ref)
    }
    print(f"  {name}: {len(rho_ref)} grid points, E_PBE = {mf_ref.e_tot:.6f}")

print(f"\nReference densities computed for: {list(ref_densities.keys())}")

In [ ]:
# Define loss function that uses actual grid density
# This computes density on the grid and compares to reference

@eqx.filter_value_and_grad
def energy_grid_density_loss(model, mols, refs, ref_densities, mol_names,
                              energy_weight=1.0, density_weight=10.0):
    '''
    Loss function combining energy error and grid density error.
    
    Runs SCF through pyscf-ad and compares the resulting density on the grid
    to a reference density (from PBE computed with pyscf-ad on same grid).
    
    :param model: XC model (RXCModel_GGA)
    :param mols: List of pyscf-ad Mole objects [H, O, H2O]
    :param refs: Dict of reference energies
    :param ref_densities: Dict of reference densities on grid (from pyscf-ad PBE)
    :param mol_names: List of molecule names matching mols order
    :param energy_weight: Weight for energy loss component
    :param density_weight: Weight for density loss component
    :return: Total loss value
    '''
    preds = []
    density_errors = []
    
    for idx, (mol, name) in enumerate(zip(mols, mol_names)):
        # Select RKS or UKS based on spin
        if mol.spin:
            mf = dft_ad.UKS(mol)
        else:
            mf = dft_ad.RKS(mol)
        
        # Configure calculation
        mf.grids.level = GRID_LEVEL
        mf.diis = False
        mf.damp = 0.5
        mf.max_cycle = 25
        
        # Set custom XC functional
        custom_eval_xc = partial(eval_xc_gga_j2, xcmodel=model)
        mf.define_xc_(custom_eval_xc, 'GGA')
        
        # Run SCF
        pred = mf.kernel()
        preds.append(pred)
        
        # Compute density on grid
        ao_val = mf._numint.eval_ao(mol, mf.grids.coords, deriv=0)
        dm = mf.make_rdm1()
        
        if dm.ndim == 3:  # UKS
            dm_total = dm[0] + dm[1]
        else:
            dm_total = dm
        
        # Compute predicted density on grid
        rho_pred = mf._numint.eval_rho(mol, ao_val, dm_total, xctype='LDA')
        
        # Get reference density and grid weights (computed with pyscf-ad, same grid)
        rho_ref = ref_densities[name]['rho']
        weights = ref_densities[name]['weights']
        
        # Compute weighted density error (integral of |rho_pred - rho_ref|^2)
        density_diff = rho_pred - rho_ref
        density_error = jnp.sum(weights * density_diff**2)
        density_errors.append(density_error)
        
        jax.debug.print("Mol {name}: E = {pred:.6f}, density_err = {derr:.8f}", 
                       name=name, pred=pred, derr=density_error)
    
    E_H, E_O, E_H2O = preds
    
    # Compute atomization energy
    pred_AE = E_H2O - 2*E_H - E_O
    ref_AE = refs['H2O_AE']
    
    # Energy loss components
    AE_loss = jnp.sqrt((pred_AE - ref_AE)**2)
    H_loss = jnp.sqrt((E_H - refs['H_TE'])**2)
    O_loss = jnp.sqrt((E_O - refs['O_TE'])**2)
    
    energy_loss_val = 50.0 * AE_loss + H_loss + O_loss
    
    # Total grid density error (sum over all molecules)
    density_loss_val = jnp.sum(jnp.array(density_errors))
    
    # Combined loss
    total_loss = energy_weight * energy_loss_val + density_weight * density_loss_val
    
    jax.debug.print("Energy loss: {el:.6f}, Density loss: {dl:.8f}, Total: {tl:.6f}",
                   el=energy_loss_val, dl=density_loss_val, tl=total_loss)
    
    return total_loss

print("Defined energy + grid density loss function")

In [ ]:
# Training with Energy + Grid Density Loss using PySCF-AD SCF
print("\n" + "="*60)
print("Training with Energy + Grid Density Loss (PySCF-AD)")
print("="*60)

# Reset model to pre-trained state
xcmodel_grid_density = RXCModel_GGA(xnet=xnet_B_trained, cnet=cnet_B_trained)

# Training configuration
TRAIN_STEPS_GRID = 25  # Reduced for debugging
ENERGY_WEIGHT_GRID = 1.0
DENSITY_WEIGHT_GRID = 10.0  # Higher weight for density

# Learning rate schedule
lr_scheduler_grid = optax.linear_schedule(
    init_value=TRAIN_LR_INIT,
    transition_steps=TRAIN_STEPS_GRID - DECAY_BEGIN,
    transition_begin=DECAY_BEGIN,
    end_value=TRAIN_LR_END,
)

# Optimizer with gradient clipping
train_optimizer_grid = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adam(learning_rate=lr_scheduler_grid)
)

# Create partial loss function with weights and reference data
energy_grid_density_loss_partial = partial(
    energy_grid_density_loss,
    ref_densities=ref_densities,
    mol_names=mol_names,
    energy_weight=ENERGY_WEIGHT_GRID,
    density_weight=DENSITY_WEIGHT_GRID
)

print(f"\nTraining with energy_weight={ENERGY_WEIGHT_GRID}, density_weight={DENSITY_WEIGHT_GRID}")
print(f"Training for {TRAIN_STEPS_GRID} steps with pyscf-ad...")
print("Using actual grid density for loss (computed with pyscf-ad)")

grid_density_trainer = Optimizer(
    model=xcmodel_grid_density,
    optim=train_optimizer_grid,
    mols=mols_ad,
    refs=refs,
    loss=energy_grid_density_loss_partial,
    print_every=5,
    steps=TRAIN_STEPS_GRID
)

# Run training
xcmodel_grid_density_trained, grid_density_losses = grid_density_trainer()

print(f"\nTraining complete!")
print(f"Final loss: {grid_density_losses[-1]:.6f}")

# Save checkpoint
grid_density_ckpt_dir = CHECKPOINT_DIRS['train_grid_density']
eqx.tree_serialise_leaves(os.path.join(grid_density_ckpt_dir, 'xcmodel_grid_density.eqx'), 
                          xcmodel_grid_density_trained)
print(f"Saved trained model to {grid_density_ckpt_dir}/xcmodel_grid_density.eqx")

### Grid Density Training with Self-Attention Networks

Training the self-attention architecture with energy + grid density loss.

In [ ]:
# Training with Energy + Grid Density Loss (Self-Attention Networks)
print("\n" + "="*60)
print("Training with Energy + Grid Density Loss (Self-Attention)")
print("="*60)

# Reset model to pre-trained self-attention state
xcmodel_grid_density_attn = RXCModel_GGA(xnet=xnet_C_trained, cnet=cnet_C_trained)

# Learning rate schedule
lr_scheduler_grid_attn = optax.linear_schedule(
    init_value=TRAIN_LR_INIT,
    transition_steps=TRAIN_STEPS_GRID - DECAY_BEGIN,
    transition_begin=DECAY_BEGIN,
    end_value=TRAIN_LR_END,
)

train_optimizer_grid_attn = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adam(learning_rate=lr_scheduler_grid_attn)
)

# Create partial loss function with weights and reference data
energy_grid_density_loss_attn = partial(
    energy_grid_density_loss,
    ref_densities=ref_densities,
    mol_names=mol_names,
    energy_weight=ENERGY_WEIGHT_GRID,
    density_weight=DENSITY_WEIGHT_GRID
)

print(f"\nTraining with energy_weight={ENERGY_WEIGHT_GRID}, density_weight={DENSITY_WEIGHT_GRID}")
print(f"Training for {TRAIN_STEPS_GRID} steps with pyscf-ad (self-attention model)...")
print("Using actual grid density for loss (computed with pyscf-ad)")

grid_density_trainer_attn = Optimizer(
    model=xcmodel_grid_density_attn,
    optim=train_optimizer_grid_attn,
    mols=mols_ad,
    refs=refs,
    loss=energy_grid_density_loss_attn,
    print_every=5,
    steps=TRAIN_STEPS_GRID
)

# Run training
xcmodel_grid_density_attn_trained, grid_density_losses_attn = grid_density_trainer_attn()

print(f"\nTraining complete!")
print(f"Final loss: {grid_density_losses_attn[-1]:.6f}")

# Save checkpoint
grid_density_attn_ckpt_dir = CHECKPOINT_DIRS['train_grid_density_attn']
eqx.tree_serialise_leaves(os.path.join(grid_density_attn_ckpt_dir, 'xcmodel_grid_density_attn.eqx'), 
                          xcmodel_grid_density_attn_trained)
print(f"Saved trained model to {grid_density_attn_ckpt_dir}/xcmodel_grid_density_attn.eqx")

In [ ]:
# Plot training progress
plt.figure(figsize=(10, 5))
plt.plot(training_losses, 'b-', linewidth=2)
plt.xlabel('Training Step')
plt.ylabel('Loss')
plt.title('Training Loss (Energy Optimization via PySCF-AD)')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Plot grid density training progress
plt.figure(figsize=(10, 5))
plt.plot(grid_density_losses, 'g-', linewidth=2)
plt.xlabel('Training Step')
plt.ylabel('Loss')
plt.title('Training Loss (Energy + Grid Density via PySCF-AD)')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

# Compare energy-only vs grid-density training
print("\n" + "="*60)
print("Comparison: Energy-only vs Energy+Grid Density Training")
print("="*60)
print(f"Energy-only final loss: {training_losses[-1]:.6f}")
print(f"Energy+Grid Density final loss: {grid_density_losses[-1]:.6f}")

In [ ]:
# Compare all training approaches (including self-attention)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left plot: Standard networks (Architecture B)
ax = axes[0]
ax.plot(training_losses, label='Energy only', alpha=0.8, linewidth=2)
ax.plot(density_losses, label='Energy + DM Reg (w=0.1)', alpha=0.8, linewidth=2)
ax.plot(density_losses_hw, label='Energy + DM Reg (w=10.0)', alpha=0.8, linewidth=2)
ax.plot(grid_density_losses, label='Energy + Grid Density', alpha=0.8, linewidth=2)
ax.set_xlabel('Training Step')
ax.set_ylabel('Loss')
ax.set_title('Training Loss - Standard Networks (Arch B)')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

# Right plot: Self-attention networks (Architecture C)
ax = axes[1]
ax.plot(training_losses_attn, label='Energy only (Attn)', alpha=0.8, linewidth=2, linestyle='--')
ax.plot(density_losses_attn, label='Energy + DM Reg (Attn)', alpha=0.8, linewidth=2, linestyle='--')
ax.plot(grid_density_losses_attn, label='Energy + Grid Density (Attn)', alpha=0.8, linewidth=2, linestyle='--')
ax.set_xlabel('Training Step')
ax.set_ylabel('Loss')
ax.set_title('Training Loss - Self-Attention Networks (Arch C)')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Combined comparison
fig, ax = plt.subplots(figsize=(14, 6))
# Standard networks
ax.plot(training_losses, label='Energy only (Std)', alpha=0.8, linewidth=2, color='blue')
ax.plot(density_losses, label='Energy + DM Reg w=0.1 (Std)', alpha=0.8, linewidth=2, color='orange')
ax.plot(grid_density_losses, label='Energy + Grid Density (Std)', alpha=0.8, linewidth=2, color='green')
# Self-attention networks
ax.plot(training_losses_attn, label='Energy only (Attn)', alpha=0.8, linewidth=2, color='blue', linestyle='--')
ax.plot(density_losses_attn, label='Energy + DM Reg (Attn)', alpha=0.8, linewidth=2, color='orange', linestyle='--')
ax.plot(grid_density_losses_attn, label='Energy + Grid Density (Attn)', alpha=0.8, linewidth=2, color='green', linestyle='--')
ax.set_xlabel('Training Step')
ax.set_ylabel('Loss')
ax.set_title('Training Loss Comparison: Standard vs Self-Attention Networks')
ax.set_yscale('log')
ax.legend(loc='upper right', ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal losses:")
print(f"{'Method':<45} {'Standard':<15} {'Self-Attention':<15}")
print("-"*75)
print(f"{'Energy only':<45} {training_losses[-1]:<15.6f} {training_losses_attn[-1]:<15.6f}")
print(f"{'Energy + DM Reg (weight=0.1)':<45} {density_losses[-1]:<15.6f} {density_losses_attn[-1]:<15.6f}")
print(f"{'Energy + DM Reg (weight=10.0)':<45} {density_losses_hw[-1]:<15.6f} {'N/A':<15}")
print(f"{'Energy + Grid Density':<45} {grid_density_losses[-1]:<15.6f} {grid_density_losses_attn[-1]:<15.6f}")

---
## Using the Trained Model with PySCF-AD

Here's how to use the trained neural network XC functional with PySCF-AD for actual DFT calculations:

In [ ]:
# Compare all trained models on H2O calculation (including self-attention)
print("\n" + "="*70)
print("Comparing All Models on H2O")
print("="*70)

def run_calculation_with_nn_xc(mol, xcmodel, label="NN-XC"):
    '''
    Run a DFT calculation using the trained neural network XC functional.
    
    :param mol: PySCF Mole object (regular, not pyscf-ad)
    :param xcmodel: Trained XC model (RXCModel_GGA)
    :param label: Label for printing
    :return: Total energy or None if failed
    '''
    try:
        # Create custom eval_xc function
        eval_xc_custom = partial(eval_xc_gga_j2, xcmodel=xcmodel)
        
        # Create DFT object
        if mol.spin == 0:
            mf = dft.RKS(mol)
        else:
            mf = dft.UKS(mol)
        
        # Set custom XC functional
        mf = mf.define_xc_(eval_xc_custom, 'GGA')
        mf.grids.level = GRID_LEVEL
        mf.verbose = 0  # Suppress output
        
        # Run calculation
        e_tot = mf.kernel()
        return e_tot
    except Exception as e:
        print(f"  {label}: FAILED - {e}")
        return None

# Create test molecule
mol_h2o = create_mol('O 0 0 0; H 0 0.757 0.587; H 0 -0.757 0.587', spin=0)

# Run PBE reference
print("\n1. PBE Reference:")
mf_pbe = run_pbe_calculation(mol_h2o)
e_pbe = mf_pbe.e_tot
print(f"   E(PBE) = {e_pbe:.6f} Ha")

# Create models at different stages of training
print("\n2. Untrained (Random) Networks:")
xnet_random = xce.net.GGA_FxNet_sigma(depth=DEPTH_B, nodes=NODES_B, seed=999)
cnet_random = xce.net.GGA_FcNet_sigma(depth=DEPTH_B, nodes=NODES_B, seed=999)
xcmodel_random = RXCModel_GGA(xnet=xnet_random, cnet=cnet_random)
e_random = run_calculation_with_nn_xc(mol_h2o, xcmodel_random, "Random")
if e_random is not None:
    print(f"   E(Random) = {e_random:.6f} Ha  (Δ = {(e_random - e_pbe)*1000:+.3f} mHa)")

print("\n--- Standard Networks (Architecture B) ---")

print("\n3. Pre-trained Networks (fitted to PBE enhancement factors):")
xcmodel_pretrained = RXCModel_GGA(xnet=xnet_B_trained, cnet=cnet_B_trained)
e_pretrained = run_calculation_with_nn_xc(mol_h2o, xcmodel_pretrained, "Pretrained")
if e_pretrained is not None:
    print(f"   E(Pretrained) = {e_pretrained:.6f} Ha  (Δ = {(e_pretrained - e_pbe)*1000:+.3f} mHa)")

print("\n4. Energy-only Trained:")
e_energy = run_calculation_with_nn_xc(mol_h2o, xcmodel_trained, "Energy-only")
if e_energy is not None:
    print(f"   E(Energy-only) = {e_energy:.6f} Ha  (Δ = {(e_energy - e_pbe)*1000:+.3f} mHa)")

print("\n5. Energy + DM Regularization (weight=0.1):")
e_dm_low = run_calculation_with_nn_xc(mol_h2o, xcmodel_density_trained, "DM-Reg(0.1)")
if e_dm_low is not None:
    print(f"   E(DM-Reg 0.1) = {e_dm_low:.6f} Ha  (Δ = {(e_dm_low - e_pbe)*1000:+.3f} mHa)")

print("\n6. Energy + DM Regularization (weight=10.0):")
e_dm_high = run_calculation_with_nn_xc(mol_h2o, xcmodel_density_trained_hw, "DM-Reg(10.0)")
if e_dm_high is not None:
    print(f"   E(DM-Reg 10.0) = {e_dm_high:.6f} Ha  (Δ = {(e_dm_high - e_pbe)*1000:+.3f} mHa)")

print("\n7. Energy + Grid Density:")
e_grid = run_calculation_with_nn_xc(mol_h2o, xcmodel_grid_density_trained, "Grid-Density")
if e_grid is not None:
    print(f"   E(Grid-Density) = {e_grid:.6f} Ha  (Δ = {(e_grid - e_pbe)*1000:+.3f} mHa)")

print("\n--- Self-Attention Networks (Architecture C) ---")

print("\n8. Pre-trained Self-Attention Networks:")
xcmodel_pretrained_attn = RXCModel_GGA(xnet=xnet_C_trained, cnet=cnet_C_trained)
e_pretrained_attn = run_calculation_with_nn_xc(mol_h2o, xcmodel_pretrained_attn, "Pretrained-Attn")
if e_pretrained_attn is not None:
    print(f"   E(Pretrained-Attn) = {e_pretrained_attn:.6f} Ha  (Δ = {(e_pretrained_attn - e_pbe)*1000:+.3f} mHa)")

print("\n9. Energy-only Trained (Self-Attention):")
e_energy_attn = run_calculation_with_nn_xc(mol_h2o, xcmodel_attn_trained, "Energy-only-Attn")
if e_energy_attn is not None:
    print(f"   E(Energy-only-Attn) = {e_energy_attn:.6f} Ha  (Δ = {(e_energy_attn - e_pbe)*1000:+.3f} mHa)")

print("\n10. Energy + DM Reg (Self-Attention, weight=0.1):")
e_dm_attn = run_calculation_with_nn_xc(mol_h2o, xcmodel_density_attn_trained, "DM-Reg-Attn")
if e_dm_attn is not None:
    print(f"   E(DM-Reg-Attn) = {e_dm_attn:.6f} Ha  (Δ = {(e_dm_attn - e_pbe)*1000:+.3f} mHa)")

print("\n11. Energy + Grid Density (Self-Attention):")
e_grid_attn = run_calculation_with_nn_xc(mol_h2o, xcmodel_grid_density_attn_trained, "Grid-Density-Attn")
if e_grid_attn is not None:
    print(f"   E(Grid-Density-Attn) = {e_grid_attn:.6f} Ha  (Δ = {(e_grid_attn - e_pbe)*1000:+.3f} mHa)")

# Summary table
print("\n" + "="*85)
print("Summary: H2O Total Energies - Standard vs Self-Attention Networks")
print("="*85)
print(f"{'Model':<40} {'Energy (Ha)':<15} {'Δ from PBE (mHa)':<15}")
print("-"*85)
print(f"{'PBE (Reference)':<40} {e_pbe:<15.6f} {'---':<15}")
print("\n--- Standard Networks (Architecture B) ---")

results_std = [
    ("Random (untrained)", e_random),
    ("Pre-trained (Fx/Fc fitting)", e_pretrained),
    ("Energy-only training", e_energy),
    ("Energy + DM Reg (w=0.1)", e_dm_low),
    ("Energy + DM Reg (w=10.0)", e_dm_high),
    ("Energy + Grid Density", e_grid),
]

for name, energy in results_std:
    if energy is not None:
        delta = (energy - e_pbe) * 1000
        print(f"{name:<40} {energy:<15.6f} {delta:<+15.3f}")
    else:
        print(f"{name:<40} {'FAILED':<15} {'---':<15}")

print("\n--- Self-Attention Networks (Architecture C) ---")

results_attn = [
    ("Pre-trained (Attn)", e_pretrained_attn),
    ("Energy-only training (Attn)", e_energy_attn),
    ("Energy + DM Reg w=0.1 (Attn)", e_dm_attn),
    ("Energy + Grid Density (Attn)", e_grid_attn),
]

for name, energy in results_attn:
    if energy is not None:
        delta = (energy - e_pbe) * 1000
        print(f"{name:<40} {energy:<15.6f} {delta:<+15.3f}")
    else:
        print(f"{name:<40} {'FAILED':<15} {'---':<15}")

### Comparing Density Matrices and Grid Densities

Beyond total energies, we compare the resulting density matrices and electron densities on the grid from each trained model. This gives insight into how different training objectives affect the predicted electronic structure.

In [ ]:
# Compare density matrices and grid densities across all trained models
print("\n" + "="*70)
print("Comparing Density Matrices and Grid Densities on H2O")
print("="*70)

def run_calculation_get_dm_and_density(mol, xcmodel, label="NN-XC"):
    '''
    Run a DFT calculation and return density matrix and density on grid.
    
    :param mol: PySCF Mole object
    :param xcmodel: Trained XC model (RXCModel_GGA) or None for PBE
    :param label: Label for printing
    :return: (energy, density_matrix, grid_density, grid_coords, grid_weights) or (None, None, None, None, None)
    '''
    try:
        # Create DFT object
        if mol.spin == 0:
            mf = dft.RKS(mol)
        else:
            mf = dft.UKS(mol)
        
        if xcmodel is not None:
            # Use neural network XC functional
            eval_xc_custom = partial(eval_xc_gga_j2, xcmodel=xcmodel)
            mf = mf.define_xc_(eval_xc_custom, 'GGA')
        else:
            # Use PBE reference
            mf.xc = REFERENCE_XC
        
        mf.grids.level = GRID_LEVEL
        mf.verbose = 0
        
        # Run calculation
        e_tot = mf.kernel()
        
        # Get density matrix
        dm = mf.make_rdm1()
        
        # Get density on grid
        ao = mf._numint.eval_ao(mol, mf.grids.coords, deriv=0)
        if dm.ndim == 3:  # UKS
            dm_total = dm[0] + dm[1]
        else:
            dm_total = dm
        rho = mf._numint.eval_rho(mol, ao, dm_total, xctype='LDA')
        
        return e_tot, dm_total, rho, mf.grids.coords, mf.grids.weights
        
    except Exception as e:
        print(f"  {label}: FAILED - {e}")
        return None, None, None, None, None

# Collect results for all models (including self-attention)
models_to_compare = [
    ("PBE (Reference)", None),
    # Standard Networks (Architecture B)
    ("Pre-trained", xcmodel_pretrained),
    ("Energy-only", xcmodel_trained),
    ("DM-Reg (w=0.1)", xcmodel_density_trained),
    ("DM-Reg (w=10.0)", xcmodel_density_trained_hw),
    ("Grid Density", xcmodel_grid_density_trained),
    # Self-Attention Networks (Architecture C)
    ("Pre-trained (Attn)", xcmodel_pretrained_attn),
    ("Energy-only (Attn)", xcmodel_attn_trained),
    ("DM-Reg (Attn)", xcmodel_density_attn_trained),
    ("Grid Density (Attn)", xcmodel_grid_density_attn_trained),
]

results_dm = {}
results_rho = {}

print("\nRunning calculations for all models...")
print("\n--- Standard Networks (Architecture B) ---")
for name, model in models_to_compare[:6]:
    e, dm, rho, coords, weights = run_calculation_get_dm_and_density(mol_h2o, model, name)
    if dm is not None:
        results_dm[name] = dm
        results_rho[name] = (rho, coords, weights)
        print(f"  {name}: converged, E = {e:.6f} Ha")
    else:
        print(f"  {name}: FAILED")

print("\n--- Self-Attention Networks (Architecture C) ---")
for name, model in models_to_compare[6:]:
    e, dm, rho, coords, weights = run_calculation_get_dm_and_density(mol_h2o, model, name)
    if dm is not None:
        results_dm[name] = dm
        results_rho[name] = (rho, coords, weights)
        print(f"  {name}: converged, E = {e:.6f} Ha")
    else:
        print(f"  {name}: FAILED")

# Reference data
dm_ref = results_dm["PBE (Reference)"]
rho_ref, coords_ref, weights_ref = results_rho["PBE (Reference)"]

print(f"\nDensity matrix shape: {dm_ref.shape}")
print(f"Grid points: {len(rho_ref)}")

In [ ]:
# Compute density matrix comparison metrics
print("\n" + "="*70)
print("Density Matrix Comparison (vs PBE Reference)")
print("="*70)

dm_metrics = {}
for name, dm in results_dm.items():
    if name == "PBE (Reference)":
        continue
    
    # Compute various metrics
    dm_diff = dm - dm_ref
    
    # Frobenius norm of difference
    frob_norm = np.linalg.norm(dm_diff, 'fro')
    
    # Max absolute difference
    max_abs_diff = np.max(np.abs(dm_diff))
    
    # Root mean square difference
    rmsd = np.sqrt(np.mean(dm_diff**2))
    
    # Trace difference (should be ~0 if electron count is conserved)
    trace_diff = np.trace(dm) - np.trace(dm_ref)
    
    dm_metrics[name] = {
        'frob_norm': frob_norm,
        'max_abs_diff': max_abs_diff,
        'rmsd': rmsd,
        'trace_diff': trace_diff
    }

# Print density matrix metrics table
print(f"\n{'Model':<25} {'Frob Norm':<12} {'Max Abs':<12} {'RMSD':<12} {'ΔTrace':<12}")
print("-"*70)
for name, metrics in dm_metrics.items():
    print(f"{name:<25} {metrics['frob_norm']:<12.6f} {metrics['max_abs_diff']:<12.6f} "
          f"{metrics['rmsd']:<12.6f} {metrics['trace_diff']:<+12.6f}")

In [ ]:
# Compute grid density comparison metrics
print("\n" + "="*70)
print("Grid Density Comparison (vs PBE Reference)")
print("="*70)

rho_metrics = {}
for name, (rho, coords, weights) in results_rho.items():
    if name == "PBE (Reference)":
        continue
    
    # Compute various metrics
    rho_diff = rho - rho_ref
    
    # Weighted integral of squared difference
    weighted_sq_diff = np.sum(weights * rho_diff**2)
    
    # Weighted integral of absolute difference
    weighted_abs_diff = np.sum(weights * np.abs(rho_diff))
    
    # Max absolute difference
    max_abs_diff = np.max(np.abs(rho_diff))
    
    # Root mean square difference (unweighted)
    rmsd = np.sqrt(np.mean(rho_diff**2))
    
    # Total electron count difference
    n_elec_diff = np.sum(weights * rho) - np.sum(weights_ref * rho_ref)
    
    rho_metrics[name] = {
        'weighted_sq_diff': weighted_sq_diff,
        'weighted_abs_diff': weighted_abs_diff,
        'max_abs_diff': max_abs_diff,
        'rmsd': rmsd,
        'n_elec_diff': n_elec_diff
    }

# Print grid density metrics table
print(f"\n{'Model':<25} {'∫|Δρ|² dV':<14} {'∫|Δρ| dV':<14} {'Max |Δρ|':<12} {'ΔN_elec':<12}")
print("-"*75)
for name, metrics in rho_metrics.items():
    print(f"{name:<25} {metrics['weighted_sq_diff']:<14.8f} {metrics['weighted_abs_diff']:<14.6f} "
          f"{metrics['max_abs_diff']:<12.6f} {metrics['n_elec_diff']:<+12.6f}")

In [ ]:
# Visualize density matrix differences
model_names = [name for name in results_dm.keys() if name != "PBE (Reference)"]
n_models = len(model_names)
n_cols = 3
n_rows = (n_models + n_cols - 1) // n_cols  # Ceiling division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
axes = axes.flatten()

for idx, name in enumerate(model_names):
    ax = axes[idx]
    dm_diff = results_dm[name] - dm_ref
    
    # Plot heatmap of difference
    vmax = np.max(np.abs(dm_diff))
    im = ax.imshow(dm_diff, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='equal')
    ax.set_title(f'{name}\nFrob norm = {dm_metrics[name]["frob_norm"]:.4f}', fontsize=10)
    ax.set_xlabel('AO index')
    ax.set_ylabel('AO index')
    plt.colorbar(im, ax=ax, label='ΔDM')

# Hide unused subplots
for idx in range(n_models, len(axes)):
    axes[idx].axis('off')

plt.suptitle('Density Matrix Differences (Model - PBE Reference)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize grid density differences
# Plot density along a line through the molecule (near the oxygen atom)

# Find grid points near the z-axis (where O and H atoms approximately are)
coords = coords_ref
x_coords = coords[:, 0]
y_coords = coords[:, 1]
z_coords = coords[:, 2]

# Select points near x=0, y=0 plane (within tolerance)
tol = 0.3
near_xz_plane = np.abs(y_coords) < tol
near_origin_x = np.abs(x_coords) < tol

# Points along z-axis near the molecule
line_mask = near_xz_plane & near_origin_x
z_line = z_coords[line_mask]
sort_idx = np.argsort(z_line)
z_sorted = z_line[sort_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Absolute densities along the line
ax1 = axes[0]
for name, (rho, _, _) in results_rho.items():
    rho_line = rho[line_mask][sort_idx]
    linestyle = '-' if name == "PBE (Reference)" else '--'
    linewidth = 2.5 if name == "PBE (Reference)" else 1.5
    ax1.plot(z_sorted, rho_line, linestyle, label=name, linewidth=linewidth)

ax1.set_xlabel('z coordinate (Bohr)')
ax1.set_ylabel('Electron density ρ(r)')
ax1.set_title('Electron Density Along z-axis (near x=0, y=0)')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Plot 2: Density differences from PBE
ax2 = axes[1]
rho_ref_line = rho_ref[line_mask][sort_idx]

for name, (rho, _, _) in results_rho.items():
    if name == "PBE (Reference)":
        continue
    rho_line = rho[line_mask][sort_idx]
    rho_diff_line = rho_line - rho_ref_line
    ax2.plot(z_sorted, rho_diff_line, label=name, linewidth=1.5)

ax2.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
ax2.set_xlabel('z coordinate (Bohr)')
ax2.set_ylabel('Δρ(r) = ρ_model - ρ_PBE')
ax2.set_title('Density Difference from PBE Along z-axis')
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Summary bar chart comparing all metrics
model_names = list(dm_metrics.keys())
n_models = len(model_names)

# Color code: blue for standard, green for self-attention
colors = ['steelblue' if '(Attn)' not in name else 'seagreen' for name in model_names]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

x = np.arange(n_models)
width = 0.7

# Plot 1: Density Matrix Frobenius Norm
ax1 = axes[0]
frob_norms = [dm_metrics[name]['frob_norm'] for name in model_names]
bars1 = ax1.bar(x, frob_norms, width, color=colors, edgecolor='black')
ax1.set_xlabel('Model')
ax1.set_ylabel('Frobenius Norm')
ax1.set_title('Density Matrix Error\n(Frobenius Norm of Difference)')
ax1.set_xticks(x)
ax1.set_xticklabels(model_names, rotation=60, ha='right', fontsize=8)
ax1.grid(True, alpha=0.3, axis='y')

# Plot 2: Grid Density Integrated Error
ax2 = axes[1]
rho_errors = [rho_metrics[name]['weighted_sq_diff'] for name in model_names]
bars2 = ax2.bar(x, rho_errors, width, color=colors, edgecolor='black')
ax2.set_xlabel('Model')
ax2.set_ylabel('∫|Δρ|² dV')
ax2.set_title('Grid Density Error\n(Integrated Squared Difference)')
ax2.set_xticks(x)
ax2.set_xticklabels(model_names, rotation=60, ha='right', fontsize=8)
ax2.grid(True, alpha=0.3, axis='y')

# Plot 3: Grid Density Max Error
ax3 = axes[2]
max_errors = [rho_metrics[name]['max_abs_diff'] for name in model_names]
bars3 = ax3.bar(x, max_errors, width, color=colors, edgecolor='black')
ax3.set_xlabel('Model')
ax3.set_ylabel('Max |Δρ|')
ax3.set_title('Grid Density Error\n(Maximum Absolute Difference)')
ax3.set_xticks(x)
ax3.set_xticklabels(model_names, rotation=60, ha='right', fontsize=8)
ax3.grid(True, alpha=0.3, axis='y')

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='steelblue', edgecolor='black', label='Standard (Arch B)'),
                   Patch(facecolor='seagreen', edgecolor='black', label='Self-Attention (Arch C)')]
fig.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(0.99, 0.99))

plt.suptitle('Summary: Density Matrix and Grid Density Errors (vs PBE)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Print final summary
print("\n" + "="*80)
print("Final Summary: Model Comparison on H2O - Standard vs Self-Attention")
print("="*80)
print("\nLower values indicate closer agreement with PBE reference.")
print("\nKey observations:")
print("- Pre-trained models should reproduce PBE well (fitted to PBE enhancement factors)")
print("- Energy-only training optimizes for total energy, may drift from PBE density")
print("- DM regularization adds penalty on density matrix, may constrain density changes")
print("- Grid density training explicitly minimizes density error on the grid")
print("- Self-attention networks may capture additional feature interactions")
print("\nCompare (Attn) variants with their standard counterparts to assess self-attention impact.")

---
## Saving and Loading Models

In [ ]:
# Save final trained models to the final_models directory
FINAL_DIR = CHECKPOINT_DIRS['final_models']

print("=" * 60)
print("Saving Standard Networks (Architecture B)")
print("=" * 60)

# Save energy-trained networks
xnet_path = os.path.join(FINAL_DIR, 'xnet_energy_trained.eqx')
eqx.tree_serialise_leaves(xnet_path, xcmodel_trained.xnet)
cnet_path = os.path.join(FINAL_DIR, 'cnet_energy_trained.eqx')
eqx.tree_serialise_leaves(cnet_path, xcmodel_trained.cnet)
print(f"Saved energy-trained networks to {FINAL_DIR}/")

# Save DM-regularization-trained networks (low weight)
xnet_dm_path = os.path.join(FINAL_DIR, 'xnet_dm_reg_trained.eqx')
eqx.tree_serialise_leaves(xnet_dm_path, xcmodel_density_trained.xnet)
cnet_dm_path = os.path.join(FINAL_DIR, 'cnet_dm_reg_trained.eqx')
eqx.tree_serialise_leaves(cnet_dm_path, xcmodel_density_trained.cnet)
print(f"Saved DM-reg-trained (low weight) networks to {FINAL_DIR}/")

# Save DM-regularization-trained networks (high weight)
xnet_dm_hw_path = os.path.join(FINAL_DIR, 'xnet_dm_reg_hw_trained.eqx')
eqx.tree_serialise_leaves(xnet_dm_hw_path, xcmodel_density_trained_hw.xnet)
cnet_dm_hw_path = os.path.join(FINAL_DIR, 'cnet_dm_reg_hw_trained.eqx')
eqx.tree_serialise_leaves(cnet_dm_hw_path, xcmodel_density_trained_hw.cnet)
print(f"Saved DM-reg-trained (high weight) networks to {FINAL_DIR}/")

# Save grid-density-trained networks
xnet_grid_path = os.path.join(FINAL_DIR, 'xnet_grid_density_trained.eqx')
eqx.tree_serialise_leaves(xnet_grid_path, xcmodel_grid_density_trained.xnet)
cnet_grid_path = os.path.join(FINAL_DIR, 'cnet_grid_density_trained.eqx')
eqx.tree_serialise_leaves(cnet_grid_path, xcmodel_grid_density_trained.cnet)
print(f"Saved grid-density-trained networks to {FINAL_DIR}/")

print("\n" + "=" * 60)
print("Saving Self-Attention Networks (Architecture C)")
print("=" * 60)

# Save pre-trained self-attention networks
xnet_pretrain_attn_path = os.path.join(FINAL_DIR, 'xnet_pretrained_attn.eqx')
eqx.tree_serialise_leaves(xnet_pretrain_attn_path, xnet_C_trained)
cnet_pretrain_attn_path = os.path.join(FINAL_DIR, 'cnet_pretrained_attn.eqx')
eqx.tree_serialise_leaves(cnet_pretrain_attn_path, cnet_C_trained)
print(f"Saved pre-trained self-attention networks to {FINAL_DIR}/")

# Save energy-trained self-attention networks
xnet_energy_attn_path = os.path.join(FINAL_DIR, 'xnet_energy_trained_attn.eqx')
eqx.tree_serialise_leaves(xnet_energy_attn_path, xcmodel_attn_trained.xnet)
cnet_energy_attn_path = os.path.join(FINAL_DIR, 'cnet_energy_trained_attn.eqx')
eqx.tree_serialise_leaves(cnet_energy_attn_path, xcmodel_attn_trained.cnet)
print(f"Saved energy-trained self-attention networks to {FINAL_DIR}/")

# Save DM-reg-trained self-attention networks
xnet_dm_attn_path = os.path.join(FINAL_DIR, 'xnet_dm_reg_trained_attn.eqx')
eqx.tree_serialise_leaves(xnet_dm_attn_path, xcmodel_density_attn_trained.xnet)
cnet_dm_attn_path = os.path.join(FINAL_DIR, 'cnet_dm_reg_trained_attn.eqx')
eqx.tree_serialise_leaves(cnet_dm_attn_path, xcmodel_density_attn_trained.cnet)
print(f"Saved DM-reg-trained self-attention networks to {FINAL_DIR}/")

# Save grid-density-trained self-attention networks
xnet_grid_attn_path = os.path.join(FINAL_DIR, 'xnet_grid_density_trained_attn.eqx')
eqx.tree_serialise_leaves(xnet_grid_attn_path, xcmodel_grid_density_attn_trained.xnet)
cnet_grid_attn_path = os.path.join(FINAL_DIR, 'cnet_grid_density_trained_attn.eqx')
eqx.tree_serialise_leaves(cnet_grid_attn_path, xcmodel_grid_density_attn_trained.cnet)
print(f"Saved grid-density-trained self-attention networks to {FINAL_DIR}/")

print(f"\nAll final models saved to: {FINAL_DIR}/")
print("\nTo load the trained networks:")
print("# Standard networks (Architecture B)")
print(f"  xnet_loaded = eqx.tree_deserialise_leaves('{xnet_path}', xnet_B)")
print(f"  cnet_loaded = eqx.tree_deserialise_leaves('{cnet_path}', cnet_B)")
print("\n# Self-attention networks (Architecture C)")
print(f"  xnet_attn_loaded = eqx.tree_deserialise_leaves('{xnet_energy_attn_path}', xnet_C)")
print(f"  cnet_attn_loaded = eqx.tree_deserialise_leaves('{cnet_energy_attn_path}', cnet_C)")

---
## Summary

This notebook demonstrated:

### Network Architectures Compared:

1. **Architecture A**: Shallow network (depth=2, nodes=8) - baseline
2. **Architecture B**: Deeper network (depth=3, nodes=16) - standard MLP
3. **Architecture C**: Deeper network with self-attention (depth=3, nodes=16, attention=True)
4. **Architecture D**: Deeper network with log-transformed inputs (depth=3, nodes=16, transform=True)
5. **Architecture E**: Deeper network with log-transforms + self-attention (depth=3, nodes=16, transform=True, attention=True)

### Training Approaches Compared:

1. **Pre-training on Enhancement Factors**: Train networks to reproduce PBE exchange (Fx) and correlation (Fc) enhancement factors
2. **Energy-Only Training**: Fine-tune on total energies using pyscf-ad autodiff through SCF
3. **Energy + Density Regularization**: Add density matrix error penalty to improve density predictions
4. **Grid Density Training**: Use actual grid density for more direct density supervision

### Log-Transform Feature (Architectures D, E):

The log-transformed networks apply numerical transformations to the input descriptors:
- Exchange: `x1 = (1 - exp(-s²)) * log(s + 1)` where s is the reduced density gradient
- Correlation: `x0 = log(rho^(1/3) + 1e-5)`, `x1 = (1 - exp(-s²)) * log(s + 1)`

These transformations improve numerical stability and can help with training dynamics.

### Self-Attention Feature (Architectures C, E):

The self-attention layer is inserted after the first hidden layer of the MLP. This allows the network to:
- Learn dynamic feature interactions between grid points
- Weight the importance of different input features adaptively
- Potentially capture non-local correlations in the density

### Key Implementation Details:

- Networks are pre-trained on enhancement factor data from reference PBE calculations
- Training uses pyscf-ad for automatic differentiation through the SCF cycle
- The `ReferenceXCModel` wrapper integrates xcquinox networks with pyscf-ad
- All models are checkpointed for reproducibility

### Key Points for Production Use:

1. Pre-training on enhancement factors provides a good initialization
2. Energy-only training is fast but may not improve density predictions
3. Density regularization helps balance energy and density accuracy
4. Grid density training provides the most direct density supervision
5. Self-attention adds flexibility but increases computational cost
6. Log-transforms can improve numerical stability for certain density ranges
